In [ ]:
# ── Cài đặt thư viện cần thiết ────────────────────────────────────────────────
# tree-sitter / tree-sitter-python : parse cây cú pháp (AST) cho Python
# gitpython                        : checkout commit của repo
# rank_bm25                        : tìm kiếm lexical (BM25)
# sentence-transformers            : sinh embedding (vector ngữ nghĩa)
# unidiff                          : đọc patch (unified diff) để lấy đáp án vàng
# datasets                         : tải benchmark SWE-bench
!pip install tree-sitter tree-sitter-python gitpython igraph tqdm rank_bm25 sentence-transformers unidiff datasets tiktoken -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 69.7 MB/s eta 0:00:00


## 1. Constants & Function Definitions
All constants and function/class definitions. Run this section once — no I/O, no side effects.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CẤU HÌNH CHẠY — đổi các hằng số ở đây rồi Run All để chạy lại             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Repositories ─────────────────────────────────────────────────────────────
# REPOS: các repo GitHub đưa vào thí nghiệm | repo_path_of(): nơi clone /content
REPOS = [
    "pallets/flask",
    "pytest-dev/pytest",
    "astropy/astropy",
]
def repo_path_of(repo):
    return f"/content/{repo.split('/')[1]}"

# Tương thích ngược cho code dùng 1 repo (mặc định = repo đầu danh sách)
REPO      = REPOS[0]
REPO_PATH = repo_path_of(REPO)

# ── Embedding model ───────────────────────────────────────────────────────────
# Model embedding nhỏ, nhanh; dùng để vector hóa node code và doc-chunk
_EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"

# Cross-encoder reranker: chấm trực tiếp cặp (query, node) — thay RRF ở tầng
# xếp hạng hop (graph = sinh ứng viên, CE = xếp hạng)
_CE_MODEL_ID = "BAAI/bge-reranker-base"

# ── SWE-bench instance selector ───────────────────────────────────────────────
# Thay đổi INSTANCE_IDX rồi Run All để chạy lại cho instance khác.
# None → dùng HEAD hiện tại (không checkout).
INSTANCE_IDX = 0

# In lại cấu hình để kiểm tra nhanh trước khi chạy phần sau
print(f"REPOS          : {', '.join(REPOS)}")
print(f"EMBED_MODEL    : {_EMBED_MODEL_ID}")
print(f"CE_RERANKER    : {_CE_MODEL_ID}")
print(f"INSTANCE_IDX   : {INSTANCE_IDX}")

REPO           : pytest-dev/pytest
REPO_PATH      : /content/pytest
EMBED_MODEL    : BAAI/bge-small-en-v1.5
INSTANCE_IDX   : 0


### 1.1 Source File Discovery
List Python files using `git ls-files`, respecting `.gitignore`.

In [ ]:
import subprocess

# ── Bộ lọc: bỏ test/example/docs và file cấu hình → graph chỉ chứa code "thật" ──
_SKIP_DIRS = {"tests", "test", "examples", "example", "benchmarks", "docs"}
_SKIP_FILES = {"conftest.py", "setup.py"}

def get_python_files(repo_path: str) -> list[str]:
    """List tracked .py files, excluding test/example directories and config files."""
    # Ưu tiên hỏi git (nhanh, đúng tập file được quản lý). Lỗi thì fallback rglob.
    try:
        out = subprocess.check_output(
            ["git", "ls-files", "--cached", "--others", "--exclude-standard"],
            cwd=repo_path, text=True, stderr=subprocess.DEVNULL,
        )
        paths = [l for l in out.splitlines() if l.endswith(".py")]
    except Exception:
        # Fallback: quét tay toàn bộ .py, bỏ thư mục .git
        paths = [
            str(p.relative_to(repo_path)).replace("\\", "/")
            for p in Path(repo_path).rglob("*.py")
            if ".git" not in p.parts
        ]

    # ── Áp bộ lọc thư mục/file lên danh sách thu được ──────────────────────────
    result = []
    for p in paths:
        parts = p.replace("\\", "/").split("/")
        # skip if any directory component is in _SKIP_DIRS
        if any(part in _SKIP_DIRS for part in parts[:-1]):
            continue
        # skip specific root-level filenames
        if parts[-1] in _SKIP_FILES:
            continue
        result.append(p)
    return result


get_python_files() defined
Skipping dirs: {'benchmarks', 'test', 'example', 'tests', 'docs', 'examples'}
Skipping files: {'conftest.py', 'setup.py'}


### 1.2 Graph Schema & Data Model
Defines `ParseContext`, `SchemaPlugin`, and `SimpleSchema` — the data model for Modules, Classes, Functions and their relationships (Defines / Calls / Imports / Inherits).

In [ ]:
from __future__ import annotations
import os
from abc import ABC, abstractmethod
from dataclasses import dataclass, field, replace as dc_replace
from typing import Callable


# ── ParseContext: "túi" trạng thái khi parse 1 file ──────────────────────────
# Giữ id module/scope cha, metadata commit/file, và các list output:
# nodes_out/edges_out (kết quả chắc chắn) + *_refs_out (ref chờ resolve cross-file).
@dataclass
class ParseContext:
    module_id:    str
    parent_id:    str
    commit:       str
    file_:        str
    directory:    str
    is_module_scope: bool
    current_class_id: str | None
    nodes_out:    list = field(default_factory=list)
    edges_out:    list = field(default_factory=list)
    call_refs_out:     list = field(default_factory=list)
    inherit_refs_out:  list = field(default_factory=list)
    instance_attr_types_out: list = field(default_factory=list)
    text_of: Callable = field(default=None, repr=False)

    @property
    def in_class(self) -> bool:
        return self.current_class_id is not None

    # Gom 1 node vào output (label + id + properties)
    def emit_node(self, label, nid, props):
        self.nodes_out.append({"labels": [label], "id": nid, "properties": props})

    # Gom 1 cạnh vào output (type + source→target, kèm field phụ tùy ý)
    def emit_edge(self, etype, src, tgt, **extra):
        entry = {"type": etype, "source": src, "target": tgt}
        entry.update(extra)
        self.edges_out.append(entry)


# ── SchemaPlugin: interface trừu tượng cho schema graph ──────────────────────
# Cho phép thay schema khác (định nghĩa node/edge nào) mà không sửa AST engine.
class SchemaPlugin(ABC):
    @abstractmethod
    def on_class(self, node, ctx): ...
    @abstractmethod
    def on_function(self, node, ctx): ...
    def on_import(self, target_module_id, ctx): pass
    @property
    @abstractmethod
    def call_edge_type(self): ...
    @property
    @abstractmethod
    def inherit_edge_type(self): ...


# ── SimpleSchema: schema thực dùng trong notebook ─────────────────────────────
# Node: Module / Class / Function | Edge: Defines / Calls / Imports / Inherits
class SimpleSchema(SchemaPlugin):
    """Module / Class / Function  +  Defines / Calls / Imports / Inherits"""

    @property
    def call_edge_type(self): return "Calls"
    @property
    def inherit_edge_type(self): return "Inherits"

    # Bắt gặp 1 định nghĩa class → tạo node Class + cạnh Defines (cha → class)
    def on_class(self, node, ctx):
        name = ctx.text_of(node.child_by_field_name("name"))
        if not name:
            return None
        cid = f"{ctx.parent_id}:{name}"   # parent_id, not module_id → correct nesting
        ctx.emit_node("Class", cid, {
            "name":       name,
            "commit":     ctx.commit,
            "file":       ctx.file_,
            "directory":  ctx.directory,
            "start_line": node.start_point[0] + 1,
            "end_line":   node.end_point[0] + 1,
            "source":     ctx.text_of(node)[:8000],   # cắt source để JSON không phình
        })
        ctx.emit_edge("Defines", ctx.parent_id, cid)
        return cid

    # Bắt gặp 1 định nghĩa function/method → tạo node Function + cạnh Defines
    def on_function(self, node, ctx):
        name = ctx.text_of(node.child_by_field_name("name"))
        if not name:
            return None
        fid = f"{ctx.parent_id}:{name}"
        ctx.emit_node("Function", fid, {
            "name":       name,
            "commit":     ctx.commit,
            "file":       ctx.file_,
            "directory":  ctx.directory,
            "start_line": node.start_point[0] + 1,
            "end_line":   node.end_point[0] + 1,
            "source":     ctx.text_of(node)[:8000],
        })
        ctx.emit_edge("Defines", ctx.parent_id, fid)
        return fid

    # Bắt gặp import đã resolve được → tạo cạnh Imports (module hiện tại → module đích)
    def on_import(self, target_module_id, ctx):
        ctx.emit_edge("Imports", ctx.module_id, target_module_id)


SCHEMA = SimpleSchema()


Schema loaded: SimpleSchema  (Module / Class / Function | Defines / Calls / Imports / Inherits)


### 1.3 Language Extractor
Configures `LanguageExtractor` for tree-sitter-python — node type mappings and field names for Python AST traversal.

In [ ]:
from dataclasses import dataclass as _dc
from tree_sitter import Language, Parser
import tree_sitter_python as tspython

# ── Khởi tạo parser tree-sitter cho Python ───────────────────────────────────
# try/except để tương thích cả API tree-sitter mới (>=0.22) lẫn cũ
try:
    _PY_LANG = Language(tspython.language())
    _parser  = Parser(_PY_LANG)
except TypeError:
    _PY_LANG = Language(tspython.language(), "python")
    _parser  = Parser()
    _parser.set_language(_PY_LANG)


# ── LanguageExtractor: "bảng tra" tên node-type/field của grammar Python ──────
# Trừu tượng hóa toàn bộ chi tiết grammar vào 1 chỗ → muốn hỗ trợ ngôn ngữ khác
# chỉ cần tạo extractor mới, không sửa ASTEngine.
@_dc(frozen=True)
class LanguageExtractor:
    parser: object
    language: object
    builtins: frozenset
    function_types: tuple
    class_types: tuple
    call_type: str
    import_type: str
    from_import_type: str
    assignment_types: tuple
    import_name_types: tuple
    aliased_import_type: str
    import_as_names_type: str
    wildcard_import_type: str
    import_keyword_type: str
    identifier_type: str
    name_field: str
    params_field: str
    superclasses_field: str
    module_name_field: str
    alias_name_field: str
    alias_alias_field: str
    call_function_field: str
    attribute_type: str
    attribute_object_field: str
    attribute_attr_field: str
    attr_name_subfield: str
    self_keywords: frozenset
    source_extensions: tuple = (".py",)
    index_file_name: str = "__init__"


# ── Cấu hình cụ thể cho Python ───────────────────────────────────────────────
EXTRACTOR = LanguageExtractor(
    parser=_parser,
    language=_PY_LANG,
    # builtins: bỏ qua khi tạo cạnh Calls (print/len/super... không phải hàm của repo)
    builtins=frozenset({
        "print", "len", "range", "str", "int", "float", "bool", "list", "dict",
        "tuple", "set", "type", "isinstance", "issubclass", "hasattr", "getattr",
        "setattr", "delattr", "super", "object", "enumerate", "zip", "map",
        "filter", "sorted", "reversed", "min", "max", "sum", "abs", "round",
        "open", "repr", "iter", "next", "any", "all", "vars", "dir", "id",
        "hash", "callable", "staticmethod", "classmethod", "property",
    }),
    # In tree-sitter-python 0.25+, `async def` is function_definition (no separate node type)
    function_types=("function_definition",),
    class_types=("class_definition",),
    call_type="call",
    import_type="import_statement",
    from_import_type="import_from_statement",
    assignment_types=("assignment", "annotated_assignment", "expression_statement"),
    import_name_types=("dotted_name", "aliased_import"),
    aliased_import_type="aliased_import",
    import_as_names_type="import_as_names",
    wildcard_import_type="wildcard_import",
    import_keyword_type="import",
    identifier_type="identifier",
    name_field="name",
    params_field="parameters",
    superclasses_field="superclasses",
    module_name_field="module_name",
    alias_name_field="name",
    alias_alias_field="alias",
    call_function_field="function",
    attribute_type="attribute",
    attribute_object_field="object",
    attribute_attr_field="attribute",
    attr_name_subfield="",
    # self_keywords: nhận diện gọi method trên chính object (self.x()/cls.x())
    self_keywords=frozenset({"self", "cls"}),
)


Extractor ready: Python / tree-sitter


### 1.4 Doc Extractor
`DocExtractor` parses `.rst` / `.md` documentation files — creates `Document` + `DocChunk` nodes with `HasChunk` edges, and collects pending `Documents` links (via `.. autofunction::` / `:func:` / `:meth:` / `:class:`) and `References` links (via `:doc:`) for resolution after code parsing.

In [ ]:
import re
from pathlib import Path

# ── Tham số bóc tách doc ──────────────────────────────────────────────────────
MAX_CHUNK_CHARS = 1600   # ~400 tokens × 4 chars
_RST_ADORN = set("=-~^*+#<>")          # ký tự gạch chân heading RST (===, ---, ~~~)

_DOC_DIR_CANDIDATES = ("docs", "doc", "documentation", "Documentation")
_DOC_SKIP = {"_build", "_static", "_templates", "_themes", "generated", "auto_examples"}


# ── Liệt kê file tài liệu (.rst/.md), bỏ thư mục build/generated ──────────────
def get_doc_files(repo_path):
    repo = Path(repo_path)
    # Lấy thư mục doc đầu tiên tồn tại trong danh sách ứng viên
    docs_dir = next(
        (repo / d for d in _DOC_DIR_CANDIDATES if (repo / d).is_dir()),
        None,
    )
    if docs_dir is None:
        return []
    out = []
    for ext in ("*.rst", "*.md"):
        for p in sorted(docs_dir.rglob(ext)):
            if any(part in _DOC_SKIP for part in p.parts):
                continue
            out.append(str(p.relative_to(repo)).replace("\\", "/"))
    return out


# ── DocExtractor: 1 file doc → node Document + nhiều DocChunk + link chờ resolve
class DocExtractor:
    # Regex bắt các directive RST trỏ tới code/doc khác:
    _ROLE_RE = re.compile(r":(?:func|meth|class|attr|exc|obj):`([^`]+)`")   # :func:`x`
    _AUTO_RE = re.compile(r"\.\.\s+auto(?:function|class|method|exception)::\s+(\S+)")  # .. autofunction::
    _DOC_RE  = re.compile(r":doc:`([^`]+)`")                                # :doc:`x` (doc→doc)
    _MOD_RE  = re.compile(r"\.\.\s+(?:current)?module::\s+(\S+)")           # .. module:: (ngữ cảnh module)

    def __init__(self, rel_path, repo_path):
        self.rel_path = rel_path
        self.abs_path = Path(repo_path) / rel_path

    # Đọc file → chia chunk theo heading → bắt link → trả nodes/edges + link chờ resolve
    def extract(self):
        try:
            text = self.abs_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return None

        lines  = text.splitlines()
        doc_id = f"doc:{self.rel_path}"
        ext    = Path(self.rel_path).suffix.lstrip(".")

        # Node Document đại diện cho cả file
        doc_node = {
            "id": doc_id, "label": "Document",
            "file": self.rel_path, "type": ext,
            "title": self._first_heading(lines) or Path(self.rel_path).stem,
        }

        nodes, edges = [doc_node], []
        pending_links, pending_refs = [], []   # link doc→code / doc→doc, resolve ở bước sau
        current_module = None                  # ngữ cảnh .. module:: hiện hành

        # Duyệt từng (heading, body) → tạo node DocChunk + cạnh HasChunk + bắt link
        for i, (heading, body) in enumerate(self._split(lines)):
            cid = f"chunk:{self.rel_path}:{i}"
            nodes.append({
                "id": cid, "label": "DocChunk",
                "file": self.rel_path, "heading": heading,
                "index": i, "text": body, "embedding": None,
            })
            edges.append({"type": "HasChunk", "source": doc_id, "target": cid})

            combined = f"{heading}\n{body}"
            # Cập nhật module ngữ cảnh nếu gặp directive .. module::
            for m in self._MOD_RE.finditer(combined):
                v = m.group(1)
                current_module = None if v == "None" else v

            # Gom các link để resolve sau (kèm module ngữ cảnh để định danh đầy đủ)
            for m in self._AUTO_RE.finditer(combined):
                pending_links.append((cid, m.group(1), current_module))
            for m in self._ROLE_RE.finditer(combined):
                pending_links.append((cid, m.group(1), current_module))
            for m in self._DOC_RE.finditer(combined):
                pending_refs.append((cid, m.group(1)))

        return {
            "doc_id": doc_id, "nodes": nodes, "edges": edges,
            "pending_links": pending_links, "pending_refs": pending_refs,
        }

    # Tìm heading đầu tiên của file (dòng có gạch chân RST ngay dưới) làm title
    def _first_heading(self, lines):
        for i in range(1, len(lines)):
            ul = lines[i].strip()
            if ul and all(c == ul[0] for c in ul) and ul[0] in _RST_ADORN:
                prev = lines[i - 1].strip()
                if prev and len(ul) >= len(prev):
                    return prev
        return None

    # Chia file thành các (heading, body) theo ranh giới heading RST
    def _split(self, lines):
        boundaries = []
        for i in range(1, len(lines)):
            ul = lines[i].strip()
            if ul and all(c == ul[0] for c in ul) and ul[0] in _RST_ADORN:
                prev = lines[i - 1].strip()
                if prev and len(ul) >= len(prev):
                    boundaries.append((i - 1, prev))

        # Không có heading nào → cả file là 1 chunk
        if not boundaries:
            body = "\n".join(lines).strip()
            return [("", body)] if body else []

        # Mỗi heading → body từ sau gạch chân tới heading kế tiếp
        chunks = []
        for idx, (pos, heading) in enumerate(boundaries):
            start = pos + 2
            end   = boundaries[idx + 1][0] if idx + 1 < len(boundaries) else len(lines)
            body  = "\n".join(lines[start:end]).strip()
            if not body:
                continue
            for chunk in self._maybe_split(heading, body):
                chunks.append(chunk)
        return chunks

    # Chunk dài quá MAX_CHUNK_CHARS → cắt nhỏ theo đoạn (giữ nguyên heading)
    def _maybe_split(self, heading, body):
        if len(body) <= MAX_CHUNK_CHARS:
            return [(heading, body)]
        parts, cur, cur_len = [], [], 0
        for para in re.split(r"\n\n+", body):
            plen = len(para)
            if cur and cur_len + plen > MAX_CHUNK_CHARS:
                parts.append((heading, "\n\n".join(cur)))
                cur, cur_len = [para], plen
            else:
                cur.append(para)
                cur_len += plen
        if cur:
            parts.append((heading, "\n\n".join(cur)))
        return parts or [(heading, body)]


# ── Bảng tên qualified_name → node_id để resolve link RST trỏ đúng node code ──
def build_name_index(code_nodes):
    """Build qualified_name -> node_id for RST role resolution (multiple keys per node)."""
    idx = {}
    for n in code_nodes:
        if n["label"] not in ("Function", "Class"):
            continue
        nid  = n.get("id", "")
        name = n.get("name", "")
        file = n.get("file", "")
        if not name:
            continue
        # Docs không document test — chặn test file khỏi index kẻo tên bị "cướp"
        # (vd pytest: Config resolve vào testing/test_assertion.py thay vì src)
        _parts = file.split("/")
        if any(p in ("test", "tests", "testing") for p in _parts) or _parts[-1].startswith("test_"):
            continue
        # Đăng ký nhiều khóa: module.name, name thuần, Class.name, pkg.Class.name
        # Bỏ tiền tố src. và đuôi .__init__ để khớp cách docs tham chiếu
        # (docs viết flask.helpers.url_for / flask.url_for, không có src.)
        file_mod = file.replace("/", ".").removesuffix(".py").removesuffix(".__init__")
        if file_mod.startswith("src."):
            file_mod = file_mod[4:]
        idx[f"{file_mod}.{name}"] = nid
        idx[f"{file_mod.split('.')[0]}.{name}"] = nid   # dạng re-export: flask.url_for
        idx[name] = nid
        try:
            after = nid.split(f":{file}:", 1)[1]
            parts = after.split(":")
            if len(parts) >= 2:
                cls = parts[-2]
                idx[f"{cls}.{name}"] = nid
                idx[f"{file_mod.split('.')[0]}.{cls}.{name}"] = nid
        except (IndexError, ValueError):
            pass
    return idx


# ── Resolve link chờ → cạnh Documents (doc→code) và References (doc→doc) ──────
def resolve_doc_links(doc_results, name_index):
    """Resolve pending RST links -> Documents + References edges (deduped)."""
    doc_ids = {r["doc_id"] for r in doc_results}
    edges, seen = [], set()

    for r in doc_results:
        # pending_links → cạnh Documents (thử name thuần → module.name → phần cuối)
        for cid, raw, cur_mod in r["pending_links"]:
            name = raw.lstrip("~.")
            target = (
                name_index.get(name)
                or (name_index.get(f"{cur_mod}.{name}") if cur_mod else None)
                or name_index.get(name.split(".")[-1])
            )
            if target:
                key = ("Documents", cid, target)
                if key not in seen:
                    edges.append({"type": "Documents", "source": cid, "target": target})
                    seen.add(key)

        # pending_refs → cạnh References (chuẩn hóa đuôi .rst, thử kèm tiền tố thư mục doc)
        for cid, ref in r["pending_refs"]:
            ref = ref.lstrip("/")
            if not ref.endswith(".rst"):
                ref += ".rst"
            tgt_candidates = [f"doc:{ref}"]
            # try with and without leading doc dir prefix
            for doc_dir in _DOC_DIR_CANDIDATES:
                if not ref.startswith(f"{doc_dir}/"):
                    tgt_candidates.append(f"doc:{doc_dir}/{ref}")
            for tgt in tgt_candidates:
                if tgt in doc_ids:
                    key = ("References", cid, tgt)
                    if key not in seen:
                        edges.append({"type": "References", "source": cid, "target": tgt})
                        seen.add(key)
                    break

    return edges


DocExtractor / get_doc_files / build_name_index / resolve_doc_links ready


### 1.4 AST Engine
`ASTEngine` parses one `.py` file in 4 passes (definitions → imports → calls → self-assigns) and maps imports for cross-file resolution.

In [ ]:
import logging
import os
from dataclasses import replace as dc_replace
from tree_sitter import Query, QueryCursor

logger = logging.getLogger(__name__)


# ── Helper: chạy 1 Query và trả {tên_capture: [Node]} (chuẩn hóa API tree-sitter)
def _caps(query, node):
    """Execute a Query and return {capture_name: [Node]} (tree-sitter 0.24+ API)."""
    result = QueryCursor(query).captures(node)
    if isinstance(result, dict):
        return result
    # Pre-0.24 fallback: list of (Node, str) tuples
    d = {}
    for n, name in result:
        d.setdefault(name, []).append(n)
    return d


class ASTEngine:
    """Parse một file .py → nodes + refs cho cross-file resolve."""

    # Khởi tạo: tính id module, chuẩn bị sẵn 4 Query (defs/imports/calls/assigns)
    def __init__(self, file_path, repo_path, base_commit, schema, extractor):
        self.extractor   = extractor
        self.file_path   = file_path
        self.repo_path   = repo_path
        self.base_commit = base_commit
        self.schema      = schema
        try:
            self.rel_path = os.path.relpath(file_path, repo_path).replace("\\", "/")
        except ValueError:
            self.rel_path = file_path.replace("\\", "/")
        self.module_id   = f"{self.base_commit}:{self.rel_path}"
        self._source: bytes = b""
        self.import_map: dict[str, str] = {}            # alias → module/symbol id
        self.from_import_map: dict[str, str | None] = {} # tên local → "module_id:tên_gốc"

        ext  = extractor
        lang = extractor.language
        self._q_defs = Query(lang,
            "\n".join(f"({t}) @def" for t in (*ext.class_types, *ext.function_types))
        )
        self._q_imports = Query(lang,
            f"({ext.import_type}) @stmt\n({ext.from_import_type}) @stmt"
        )
        self._q_calls   = Query(lang, f"({ext.call_type}) @call")
        self._q_assigns = Query(lang, "(assignment) @assign")

    # ── Public ─────────────────────────────────────────────────────────────────

    # Điểm vào: đọc file → parse → chạy 4 pass → trả dict kết quả cho resolve sau
    def parse(self):
        try:
            with open(self.file_path, "rb") as f:
                self._source = f.read()
        except Exception:
            return None
        try:
            tree = self.extractor.parser.parse(self._source)
        except Exception:
            return None

        commit, _, file_ = self.module_id.partition(":")
        ctx = ParseContext(
            module_id=self.module_id, parent_id=self.module_id,
            commit=commit, file_=file_,
            directory=os.path.dirname(file_) or ".",
            is_module_scope=True, current_class_id=None,
            text_of=self._text,
        )
        ctx.emit_node("Module", self.module_id, {"name": self.rel_path, "commit": self.base_commit})

        # 4 pass bóc tách trên cùng cây cú pháp
        self._extract_definitions(tree, ctx)
        self._extract_imports(tree, ctx)
        self._extract_calls(tree, ctx)
        self._extract_self_assigns(tree, ctx)

        # local_scope/class_scopes phục vụ resolve call cross-file
        local_scope, class_scopes = self._build_scopes(ctx.nodes_out)
        return {
            "module_id":    self.module_id,
            "nodes":        ctx.nodes_out,
            "definite_edges":      ctx.edges_out,
            "call_refs":           ctx.call_refs_out,
            "inherit_refs":        ctx.inherit_refs_out,
            "instance_attr_types": ctx.instance_attr_types_out,
            "import_map":          self.import_map,
            "from_import_map":     self.from_import_map,
            "local_scope":         local_scope,
            "class_scopes":        class_scopes,
        }

    # ── Scope helpers ──────────────────────────────────────────────────────────

    # Leo node.parent → id scope CHỨA node (vd "commit:file:Class:method")
    def _scope_of(self, node):
        """ID of the scope that CONTAINS node (walks up node.parent chain)."""
        ext, parts, cur = self.extractor, [], node.parent
        while cur is not None:
            if cur.type in (*ext.class_types, *ext.function_types):
                n = cur.child_by_field_name(ext.name_field)
                if n:
                    parts.append(self._text(n))
            cur = cur.parent
        return ":".join([self.module_id] + list(reversed(parts))) if parts else self.module_id

    # Leo node.parent → id class bao quanh gần nhất (hoặc None)
    def _class_of(self, node):
        """ID of the innermost class enclosing node, or None."""
        ext, cur = self.extractor, node.parent
        while cur is not None:
            if cur.type in ext.class_types:
                n = cur.child_by_field_name(ext.name_field)
                if n:
                    return f"{self._scope_of(cur)}:{self._text(n)}"
            cur = cur.parent
        return None

    # ── Pass 1: class + function definitions ──────────────────────────────────

    # Tìm mọi class/function (sắp theo vị trí) → tạo node; với class còn lấy superclass
    def _extract_definitions(self, tree, ctx):
        ext = self.extractor
        def_nodes = sorted(_caps(self._q_defs, tree.root_node).get("def", []),
                           key=lambda n: n.start_byte)
        for node in def_nodes:
            tmp = dc_replace(ctx, parent_id=self._scope_of(node),
                             current_class_id=self._class_of(node))
            if node.type in ext.class_types:
                cid = self.schema.on_class(node, tmp)
                if cid:
                    # Bóc danh sách lớp cha → inherit_refs (chờ resolve thành cạnh Inherits)
                    supers = node.child_by_field_name(ext.superclasses_field)
                    if supers:
                        for arg in supers.children:
                            base = self._extract_base_name(arg)
                            if base:
                                ctx.inherit_refs_out.append({"class_id": cid, "base_name": base})
            else:
                self.schema.on_function(node, tmp)

    # ── Pass 2: imports ────────────────────────────────────────────────────────

    # Xử lý `import x` và `from a import b` → build import_map/from_import_map + cạnh Imports
    def _extract_imports(self, tree, ctx):
        ext = self.extractor
        for node in _caps(self._q_imports, tree.root_node).get("stmt", []):
            if node.type == ext.import_type:
                # import a.b as c → tách module + alias, resolve sang file thật
                for sub in node.children:
                    if sub.type in ext.import_name_types:
                        raw = self._text(sub)
                        mod_str, _, alias = raw.partition(" as ")
                        mod_str = mod_str.strip()
                        alias   = alias.strip() if alias.strip() else mod_str.split(".")[-1]
                        tid = self._resolve_module(mod_str)
                        if tid:
                            self.schema.on_import(tid, ctx)
                            self.import_map[alias] = tid
                            if "." in mod_str:
                                self.import_map[mod_str] = tid
            elif node.type == ext.from_import_type:
                # from mod import a, b → cố resolve từng symbol; không được thì ghi nhớ để resolve sau
                mod_node = node.child_by_field_name(ext.module_name_field)
                if not mod_node:
                    continue
                mod_str    = self._text(mod_node)
                module_tid = self._resolve_module(mod_str)
                if module_tid:
                    self.schema.on_import(module_tid, ctx)
                for local_name, orig_name in self._get_from_imports(node):
                    sub_tid = self._resolve_module(f"{mod_str}.{orig_name}")
                    if sub_tid:
                        self.import_map[local_name] = sub_tid
                    elif module_tid:
                        self.from_import_map[local_name] = f"{module_tid}:{orig_name}"
                    else:
                        self.from_import_map[local_name] = None

    # Lấy danh sách (tên_local, tên_gốc) trong mệnh đề `from ... import ...`
    def _get_from_imports(self, from_stmt):
        ext, results, past_import = self.extractor, [], False
        for child in from_stmt.children:
            if child.type == ext.import_keyword_type:
                past_import = True; continue
            if not past_import:
                continue
            if child.type == ext.wildcard_import_type:   # from x import *  → bỏ qua
                break
            if child.type == ext.identifier_type:
                n = self._text(child); results.append((n, n))
            elif child.type == ext.import_as_names_type:
                for sub in child.children:
                    if sub.type == ext.identifier_type:
                        n = self._text(sub); results.append((n, n))
                    elif sub.type == ext.aliased_import_type:
                        r = self._resolve_aliased(sub)
                        if r: results.append(r)
            elif child.type == ext.aliased_import_type:
                r = self._resolve_aliased(child)
                if r: results.append(r)
        return results

    # `orig as alias` → (alias, orig)
    def _resolve_aliased(self, node):
        ext   = self.extractor
        orig  = node.child_by_field_name(ext.alias_name_field)
        alias = node.child_by_field_name(ext.alias_alias_field)
        if orig:
            o = self._text(orig)
            return (self._text(alias) if alias else o, o)
        return None

    # ── Pass 3: call references ────────────────────────────────────────────────

    # Phân loại mọi lời gọi thành kind (simple/attr/self_method/self_attr_method) → call_refs
    def _extract_calls(self, tree, ctx):
        ext = self.extractor
        for call_node in _caps(self._q_calls, tree.root_node).get("call", []):
            func = call_node.child_by_field_name(ext.call_function_field)
            if func is None:
                continue
            base = {
                "caller_id": self._scope_of(call_node),   # ai gọi
                "line":      call_node.start_point[0] + 1,
                "file":      self.rel_path,
            }
            if func.type == ext.identifier_type:
                # foo()  → simple (bỏ qua builtins)
                name = self._text(func)
                if name not in ext.builtins:
                    ctx.call_refs_out.append({**base, "kind": "simple", "name": name})

            elif func.type == ext.attribute_type:
                # obj.attr() → phân biệt self.x() / self.attr.x() / obj.x()
                obj  = func.child_by_field_name(ext.attribute_object_field)
                attr = func.child_by_field_name(ext.attribute_attr_field)
                if not (obj and attr):
                    continue
                attr_name = self._text(attr)
                if attr_name in ext.builtins:
                    continue
                obj_text = self._text(obj)
                if obj_text in ext.self_keywords:
                    # self.method() → resolve trong class hiện tại
                    ctx.call_refs_out.append({
                        **base, "kind": "self_method", "name": attr_name,
                        "class_id": self._class_of(call_node),
                    })
                elif obj.type == ext.attribute_type:
                    inner_obj  = obj.child_by_field_name(ext.attribute_object_field)
                    inner_attr = obj.child_by_field_name(ext.attribute_attr_field)
                    if inner_obj and inner_attr and self._text(inner_obj) in ext.self_keywords:
                        # self.attr.method() → cần biết kiểu của self.attr (xem Pass 4)
                        ctx.call_refs_out.append({
                            **base, "kind": "self_attr_method", "name": attr_name,
                            "attr": self._text(inner_attr),
                            "class_id": self._class_of(call_node),
                        })
                    else:
                        ctx.call_refs_out.append({**base, "kind": "attr", "name": attr_name, "obj": obj_text})
                else:
                    # obj.method() → resolve qua import_map của obj
                    ctx.call_refs_out.append({**base, "kind": "attr", "name": attr_name, "obj": obj_text})

    # ── Pass 4: self.attr = Type() assignments ────────────────────────────────

    # Bắt `self.attr = Type()` → ghi kiểu của thuộc tính, để resolve self.attr.method()
    def _extract_self_assigns(self, tree, ctx):
        ext = self.extractor
        for node in _caps(self._q_assigns, tree.root_node).get("assign", []):
            left  = node.child_by_field_name("left")
            right = node.child_by_field_name("right")
            if not (left and right and left.type == ext.attribute_type and right.type == ext.call_type):
                continue
            obj_node  = left.child_by_field_name(ext.attribute_object_field)
            attr_node = left.child_by_field_name(ext.attribute_attr_field)
            if not (obj_node and attr_node):
                continue
            if self._text(obj_node) not in ext.self_keywords:
                continue
            # Lấy tên kiểu ở vế phải (Type() hoặc mod.Type())
            func = right.child_by_field_name(ext.call_function_field)
            if not func:
                continue
            if func.type == ext.identifier_type:
                type_name = self._text(func)
            elif func.type == ext.attribute_type:
                a = func.child_by_field_name(ext.attribute_attr_field)
                type_name = self._text(a) if a else None
            else:
                continue
            if not type_name or type_name in ext.builtins:
                continue
            class_id = self._class_of(obj_node)
            if class_id:
                ctx.instance_attr_types_out.append({
                    "class_id": class_id,
                    "attr":     self._text(attr_node),
                    "type_name": type_name,
                })

    # ── Module path resolver ───────────────────────────────────────────────────

    # Tên module (kể cả relative `..pkg`) → đường dẫn file thật trong repo, hoặc None
    def _resolve_module(self, module):
        if module.startswith("."):
            # relative import: số dấu chấm = số cấp đi lên
            dots     = len(module) - len(module.lstrip("."))
            rel_part = module.lstrip(".")
            base_dir = os.path.dirname(self.rel_path)
            for _ in range(dots - 1):
                base_dir = os.path.dirname(base_dir)
            parts = (
                os.path.join(base_dir, rel_part.replace(".", os.sep)).replace("\\", "/")
                if rel_part else base_dir
            )
        else:
            parts = module.replace(".", "/")
        # Thử cả file.py lẫn package/__init__.py
        init_name = self.extractor.index_file_name
        for src_ext in self.extractor.source_extensions:
            for candidate in (f"{parts}{src_ext}", f"{parts}/{init_name}{src_ext}"):
                if os.path.exists(os.path.join(self.repo_path, candidate)):
                    return f"{self.base_commit}:{os.path.relpath(os.path.join(self.repo_path, candidate), self.repo_path).replace(chr(92), '/')}"
        return None

    # ── Scope builder ──────────────────────────────────────────────────────────

    # Từ node của file → 2 bảng: local_scope (tên top-level) & class_scopes (method theo class)
    def _build_scopes(self, nodes):
        local_scope, class_scopes = {}, {}
        for node in nodes:
            label   = node["labels"][0]
            if label == "Module":
                continue
            node_id = node["id"]
            name    = node["properties"].get("name", "")
            parts   = node_id[len(self.module_id):].lstrip(":").split(":")
            if len(parts) == 1:
                local_scope[name] = node_id
            elif len(parts) == 2 and label == "Function":
                class_id = f"{self.module_id}:{parts[0]}"
                class_scopes.setdefault(class_id, {})[name] = node_id
        return local_scope, class_scopes

    # ── Helpers ────────────────────────────────────────────────────────────────

    # Lấy text gốc của 1 node từ byte buffer
    def _text(self, node):
        return self._source[node.start_byte:node.end_byte].decode("utf-8", errors="replace")

    # Lấy tên lớp cơ sở từ node superclass (xử lý identifier / attribute / subscript)
    def _extract_base_name(self, node):
        ext = self.extractor
        if node.type == ext.identifier_type:
            return self._text(node)
        if node.type == ext.attribute_type:
            a = node.child_by_field_name(ext.attribute_attr_field)
            return self._text(a) if a else ""
        if node.type in ("subscript", "type"):
            child = node.child_by_field_name("value") or (node.children[0] if node.child_count else None)
            return self._extract_base_name(child) if child else ""
        return ""

    # ── Cross-file resolution — returns ONLY semantic (Calls, Inherits) edges ──

    # Sau khi parse hết file: gộp scope/import toàn repo → resolve thành cạnh Calls/Inherits
    @staticmethod
    def resolve_cross_file(all_results, schema):
        from collections import defaultdict
        # Tập id node toàn cục + các bảng tra theo module
        all_node_ids = set()
        all_node_ids.update(n["id"] for r in all_results for n in r["nodes"])
        fmap   = {r["module_id"]: r["from_import_map"] for r in all_results}
        lscope = {r["module_id"]: r["local_scope"]     for r in all_results}
        cscope = {r["module_id"]: r["class_scopes"]    for r in all_results}
        imap   = {r["module_id"]: r["import_map"]      for r in all_results}
        iat    = {r["module_id"]: r["instance_attr_types"] for r in all_results}

        # Build global method map: lowercase_method_name -> list of candidate node IDs
        global_method_map = defaultdict(list)
        for r in all_results:
            for node in r["nodes"]:
                if node["labels"][0] in ("Function", "Class"):
                    name = node["properties"].get("name", "")
                    if name:
                        global_method_map[name.lower()].append(node["id"])

        semantic_edges = []   # Calls + Inherits only

        # Inherit refs → cạnh Inherits (thử local scope → import → from-import)
        for r in all_results:
            mid   = r["module_id"]
            ls, im, fim = lscope.get(mid, {}), imap.get(mid, {}), fmap.get(mid, {})
            for ref in r["inherit_refs"]:
                cid, base = ref["class_id"], ref["base_name"]
                target = (
                    ls.get(base)
                    or im.get(base)
                    or (fim.get(base) if fim.get(base) in all_node_ids else None)
                )
                if target and target in all_node_ids:
                    semantic_edges.append({"type": schema.inherit_edge_type, "source": cid, "target": target})

        # Call refs → cạnh Calls (mỗi ref gọi _resolve_call; target có thể là list)
        for r in all_results:
            mid  = r["module_id"]
            ls   = lscope.get(mid, {})
            im   = imap.get(mid, {})
            fim  = fmap.get(mid, {})
            cs   = cscope.get(mid, {})
            iat_ = {e["class_id"]: e for e in iat.get(mid, [])}

            for ref in r["call_refs"]:
                target = ASTEngine._resolve_call(
                    ref, all_node_ids, ls, im, fim, cs, iat_, cs, lscope, imap, global_method_map
                )
                if target:
                    if isinstance(target, list):
                        for t in target:
                            semantic_edges.append({
                                "type": schema.call_edge_type,
                                "source": ref["caller_id"],
                                "target": t,
                                "line": ref.get("line"),
                            })
                    else:
                        semantic_edges.append({
                            "type": schema.call_edge_type,
                            "source": ref["caller_id"],
                            "target": target,
                            "line": ref.get("line"),
                        })
        return semantic_edges

    # Resolve 1 lời gọi → id node đích, theo từng kind; cuối cùng có fallback heuristic
    @staticmethod
    def _resolve_call(ref, all_ids, ls, im, fim, cs, iat, cscope, lscope, imap, global_method_map=None):
        kind = ref.get("kind", "simple")
        name = ref.get("name", "")

        if kind == "simple":
            # foo() → local scope / import / from-import
            resolved = ASTEngine._find_first(name, all_ids, ls, im, fim)
            if resolved:
                return resolved

        if kind == "self_method":
            # self.foo() → method cùng class
            class_id = ref.get("class_id")
            if class_id:
                target = cs.get(class_id, {}).get(name)
                if target and target in all_ids:
                    return target

        if kind == "self_attr_method":
            # self.attr.foo() → tra kiểu của self.attr (Pass 4) rồi tìm method trong class kiểu đó
            class_id = ref.get("class_id")
            attr     = ref.get("attr", "")
            if class_id and attr in iat:
                type_name    = iat[attr]["type_name"]
                target_class = ASTEngine._find_first(type_name, all_ids, ls, im, fim)
                if target_class:
                    t = cscope.get(target_class, {}).get(name)
                    if t and t in all_ids:
                        return t

        if kind == "attr":
            # obj.foo() → obj là module đã import → tìm trong scope module đó
            obj = ref.get("obj", "")
            module_tid = im.get(obj) or fim.get(obj)
            if module_tid:
                remote_ls = lscope.get(module_tid, {})
                t = remote_ls.get(name)
                if t and t in all_ids:
                    return t
                candidate = imap.get(module_tid, {}).get(name)
                if candidate and candidate in all_ids:
                    return candidate

        # Heuristic-based fallback: tên method hiếm (≤3 ứng viên, không phổ biến) → nối luôn
        if global_method_map and name:
            common_methods = {
                "get", "set", "run", "update", "delete", "clear", "close", "save", "load",
                "__init__", "__str__", "__repr__", "__call__", "setup", "teardown", "execute",
                "pop", "append", "extend", "insert", "remove", "add", "format", "keys", "values", "items"
            }
            if name not in common_methods:
                candidates = global_method_map.get(name.lower(), [])
                if 0 < len(candidates) <= 3:
                    valid_candidates = [c for c in candidates if c in all_ids]
                    if valid_candidates:
                        return valid_candidates

        return None

    # Tra tên lần lượt qua (local scope, import_map, from_import_map) → id hợp lệ đầu tiên
    @staticmethod
    def _find_first(name, all_ids, ls, im, fim):
        for source in (ls, im, fim):
            t = source.get(name)
            if t and t in all_ids:
                return t
        return None


ASTEngine ready: query-based (Query + QueryCursor, tree-sitter 0.24+)


### 1.5 Build & Search Pipeline
Three functions that form the core pipeline:
- `build_graph(commit)` — checkout, parse, embed (BGE-small), export JSON (disk-cached)
- `load_graph_indices(path)` — load JSON, build BM25 + vector + call-graph indices (memory-cached)
- `get_oracle_nodes(patch, nodes)` — map gold patch hunks to graph nodes (ground truth)

In [ ]:
import json
import numpy as np
from collections import deque, defaultdict
from pathlib import Path
from datetime import datetime, timezone
from git import Repo as _Repo
from rank_bm25 import BM25Okapi
import hashlib

# ── Cache trong phiên: model / index đã build / embedding theo nội dung ───────
_MODEL_CACHE = {}
_INDEX_CACHE = {}
_EMB_CACHE   = {}

# Load model embedding 1 lần duy nhất cho mỗi model_id
def _get_model(model_id=_EMBED_MODEL_ID):
    if model_id not in _MODEL_CACHE:
        from sentence_transformers import SentenceTransformer
        _MODEL_CACHE[model_id] = SentenceTransformer(model_id)
    return _MODEL_CACHE[model_id]

# Load cross-encoder reranker 1 lần (tự chọn GPU nếu có; max 512 token/cặp)
def _get_ce(model_id=_CE_MODEL_ID):
    if model_id not in _MODEL_CACHE:
        from sentence_transformers import CrossEncoder
        _MODEL_CACHE[model_id] = CrossEncoder(model_id, max_length=512)
    return _MODEL_CACHE[model_id]

# Text dùng để embed 1 node code: 2000 ký tự đầu của source (fallback: label+name)
def _sig_doc_fn(node):
    src = node.get("source", "")
    return src[:2000] if src else f"{node.get('label','')} {node.get('name','')}"

# Embed có cache theo sha1(text): source giống nhau giữa các commit chỉ encode 1 lần
def encode_cached(texts, model, batch_size=64):
    """Embed texts with an in-session cache keyed by sha1(text).
    Source giống hệt giữa các commit chỉ encode 1 lần (~80-90% reuse trên cùng repo)."""
    if not texts:
        return np.zeros((0, model.get_sentence_embedding_dimension()), dtype=np.float32)
    keys = [hashlib.sha1(t.encode("utf-8")).hexdigest() for t in texts]
    # Tách phần đã có trong cache (out) và phần thiếu (miss) cần encode
    out, miss_i, miss_t = [None] * len(texts), [], []
    for i, k in enumerate(keys):
        v = _EMB_CACHE.get(k)
        if v is None:
            miss_i.append(i); miss_t.append(texts[i])
        else:
            out[i] = v
    # Chỉ encode phần thiếu rồi nạp ngược vào cache + vị trí tương ứng
    if miss_t:
        vecs = model.encode(miss_t, batch_size=batch_size,
                            normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
        for j, i in enumerate(miss_i):
            _EMB_CACHE[keys[i]] = vecs[j]
            out[i] = vecs[j]
    return np.vstack(out)


Globals ready: _MODEL_CACHE, _INDEX_CACHE, _get_model, _sig_doc_fn


In [ ]:
# ── build_graph(): orchestrator — build full graph cho 1 commit ──────────────
# Luồng 7 bước: cache → checkout → parse AST → resolve → flatten → doc → embed → export
import time

def build_graph(commit, repo_path=REPO_PATH, output_dir="/content", force=False,
                embed_model_id=_EMBED_MODEL_ID):
    """
    Checkout commit → parse AST + docs → embed → export JSON.
    Always builds the full graph (code nodes + Document/DocChunk nodes).
    """
    repo_name   = Path(repo_path).name
    short       = commit[:8]
    output_file = f"{output_dir}/{repo_name}_graph_{short}.json"

    # 0. Disk cache: đã có JSON đúng schema "simple+doc" thì trả luôn, khỏi build lại
    if not force and Path(output_file).exists():
        try:
            with open(output_file) as _f:
                _schema = json.load(_f).get("meta", {}).get("schema", "")
            if _schema == "simple+doc":
                print(f"[{short}] disk cache hit → {output_file}", flush=True)
                return output_file
            print(f"[{short}] cached schema={_schema!r} — rebuilding with doc layer", flush=True)
        except Exception:
            pass

    # 1. Checkout
    _t0 = time.perf_counter()
    _repo = _Repo(repo_path)
    _repo.git.checkout(commit)
    _t1 = time.perf_counter()

    # 2. Source files
    src_files = get_python_files(repo_path)

    # 3. AST parse — File-level Cache (Bỏ qua Tree-sitter cho 99% file không đổi)
    global _FILE_AST_CACHE
    if "_FILE_AST_CACHE" not in globals():
        _FILE_AST_CACHE = {}
    import hashlib

    results, n_cache, n_parse, n_fail = [], 0, 0, 0
    for rel in src_files:
        try:
            fpath = Path(repo_path) / rel
            content = fpath.read_bytes()
            fhash = hashlib.sha1(content).hexdigest()
            
            if fhash in _FILE_AST_CACHE:
                old_short, cached_r = _FILE_AST_CACHE[fhash]
                if old_short == short:
                    results.append(cached_r)
                else:
                    # Đổi tên commit trên ID node bằng cách serialize siêu tốc
                    # (không import json tại đây — import trong hàm biến 'json'
                    #  thành local toàn hàm → UnboundLocalError ở chỗ dùng khác)
                    r_str = json.dumps(cached_r)
                    r_str = r_str.replace(f"{old_short}:", f"{short}:").replace(f'"{old_short}"', f'"{short}"')
                    results.append(json.loads(r_str))
                n_cache += 1
                continue
                
            eng = ASTEngine(str(fpath), repo_path, short, SCHEMA, EXTRACTOR)
            r = eng.parse()
            if r:
                _FILE_AST_CACHE[fhash] = (short, r)
                results.append(r)
                n_parse += 1
        except Exception:
            n_fail += 1
    # failed > 0 → file bị bỏ khỏi graph → oracle/recall thiếu mà không có dấu vết
    print(f"[{short}] AST: {len(results)}/{len(src_files)} files "
          f"(cache={n_cache} parsed={n_parse} failed={n_fail})", flush=True)

    # 4. Resolve cross-file edges (sem = Calls/Inherits, def = Defines/Imports)
    sem_edges = ASTEngine.resolve_cross_file(results, SCHEMA)
    def_edges = [e for r in results for e in r["definite_edges"]]

    # 5. Flatten code nodes / edges (dedupe node theo id)
    all_nodes = list({
        node["id"]: {
            "id":         node["id"],
            "label":      node["labels"][0],
            "name":       node["properties"].get("name", ""),
            "file":       node["properties"].get("file", ""),
            "start_line": node["properties"].get("start_line"),
            "end_line":   node["properties"].get("end_line"),
            "source":     node["properties"].get("source", ""),
            "community":  node["properties"].get("community"),
        }
        for r in results for node in r["nodes"]
    }.values())
    all_edges = [
        {"type": e["type"], "source": e["source"], "target": e["target"], "line": e.get("line")}
        for e in (def_edges + sem_edges)
    ]

    _t2 = time.perf_counter()

    # 5b. Doc parse → thêm node Document/DocChunk + File-level Cache
    global _FILE_DOC_CACHE
    if "_FILE_DOC_CACHE" not in globals():
        _FILE_DOC_CACHE = {}
        
    doc_files = get_doc_files(repo_path)
    doc_results = []
    for rel in doc_files:
        try:
            fpath = Path(repo_path) / rel
            content = fpath.read_bytes()
            fhash = hashlib.sha1(content).hexdigest()
            if fhash in _FILE_DOC_CACHE:
                doc_results.append(_FILE_DOC_CACHE[fhash])
                continue
                
            r = DocExtractor(rel, repo_path).extract()
            if r:
                _FILE_DOC_CACHE[fhash] = r
                doc_results.append(r)
        except Exception:
            pass
    doc_nodes   = [n for r in doc_results for n in r["nodes"]]
    doc_edges   = [e for r in doc_results for e in r["edges"]]
    cross_edges = resolve_doc_links(doc_results, build_name_index(all_nodes))
    n_has_chunk = sum(1 for e in doc_edges   if e["type"] == "HasChunk")
    n_documents = sum(1 for e in cross_edges if e["type"] == "Documents")
    n_refs      = sum(1 for e in cross_edges if e["type"] == "References")
    n_pending   = sum(len(r["pending_links"]) for r in doc_results)
    # % resolved = sức khỏe cầu doc→code (đo trực tiếp hiệu quả build_name_index)
    print(f"[{short}] docs: {len(doc_nodes)} nodes | HasChunk={n_has_chunk}  "
          f"Documents={n_documents}/{n_pending} ({n_documents/max(1, n_pending):.0%} resolved)  "
          f"References={n_refs}", flush=True)
    all_nodes += doc_nodes
    all_edges += doc_edges + cross_edges

    _t3 = time.perf_counter()

    # 6. Embeddings — Function/Class: sig+docstring; DocChunk: heading+text
    _model    = _get_model(embed_model_id)
    vec_cands = [n for n in all_nodes if n["label"] in ("Function", "Class", "DocChunk")]
    embed_texts = [
        _sig_doc_fn(n) if n["label"] in ("Function", "Class")
        else (f"{n['heading']}\n{n['text']}" if n.get('heading') else n.get('text', ""))
        for n in vec_cands
    ]
    vecs = encode_cached(embed_texts, _model, batch_size=64)  # in-session cache
    id2emb = {vec_cands[i]["id"]: vecs[i].tolist() for i in range(len(vec_cands))}
    for node in all_nodes:
        node["embedding"] = id2emb.get(node["id"])

    _t4 = time.perf_counter()

    # 7. Export ra JSON (meta + nodes + edges) + dọn index cache cũ
    try:
        remote_url = _repo.remotes.origin.url
    except Exception:
        remote_url = repo_path

    graph = {
        "meta": {
            "repo": remote_url, "commit": short, "schema": "simple+doc", "lang": "python",
            "built_at": datetime.now(timezone.utc).isoformat(),
            "node_count": len(all_nodes), "edge_count": len(all_edges),
            "embed_model": embed_model_id, "embed_dim": int(vecs.shape[1]),
        },
        "nodes": all_nodes, "edges": all_edges,
    }
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(graph, f, ensure_ascii=False)

    _INDEX_CACHE.pop(output_file, None)

    size_mb = Path(output_file).stat().st_size / 1024 / 1024
    print(f"[{short}] → {output_file}  ({len(all_nodes)} nodes, {len(all_edges)} edges, {size_mb:.1f} MB) | "
          f"checkout={_t1-_t0:.1f}s parse={_t2-_t1:.1f}s doc={_t3-_t2:.1f}s "
          f"embed={_t4-_t3:.1f}s export={time.perf_counter()-_t4:.1f}s", flush=True)
    return output_file


build_graph() ready


In [ ]:
import re
import time
from tqdm.auto import tqdm
_CAMEL_RE = re.compile(r'([A-Z])')
_ALPHANUM_RE = re.compile(r'[a-zA-Z0-9]+')
def _tokenize_code(txt):
    # Dùng regex đã pre-compile để tăng tốc tối đa
    return _ALPHANUM_RE.findall(_CAMEL_RE.sub(r' \1', txt).lower())

# ── _embed_query(): làm sạch query trước khi encode ──────────────────────────
# bge-small cắt input ở 512 token; issue dài kèm code block chiếm hết cửa sổ →
# bỏ code block, giữ ~2000 chars đầu (≈512 token) phần mô tả tự nhiên.
_MD_CODEBLOCK_RE = re.compile(r"```.*?```", re.S)
def _embed_query(q):
    return _MD_CODEBLOCK_RE.sub(" ", q)[:2000]

# ── load_graph_nodes(): loader nhẹ — chỉ nodes + map id, không build index ───
# Dùng để trích oracle mà không tốn chi phí dựng index tìm kiếm.
def load_graph_nodes(output_file):
    """Lightweight loader — nodes + node_by_id only, no index built.
    Used for oracle extraction without the full index cost."""
    with open(output_file, "r", encoding="utf-8") as f:
        gdata = json.load(f)
    nodes = gdata["nodes"]
    return {"nodes": nodes, "node_by_id": {n["id"]: n for n in nodes}}


# ── load_graph_indices(): nạp graph + dựng toàn bộ index truy hồi ────────────
def load_graph_indices(output_file):
    """
    Load graph JSON, build retrieval indices.

    Two search indices:
      search()      — BM25 + vector on AST nodes only (Function/Class/Module)
      search_full() — BM25 + vector on AST nodes + DocChunk nodes

    Two coverage traversals (pure BFS, no decay, no rerank, return ALL reachable nodes):
      hop_coverage_ast()  — seeds via search()      + BFS via AST edges only
      hop_coverage_full() — coverage AST ∪ node có doc_votes() (doc-as-signal)
    """
    # Memory cache: đã dựng index cho file này thì trả luôn (im lặng — bị gọi lặp)
    if output_file in _INDEX_CACHE:
        return _INDEX_CACHE[output_file]

    _stem = Path(output_file).stem
    _t0 = time.perf_counter()
    with open(output_file, "r", encoding="utf-8") as f:
        gdata = json.load(f)
    _t_load = time.perf_counter()

    nodes          = gdata["nodes"]
    embed_model_id = gdata["meta"].get("embed_model", _EMBED_MODEL_ID)
    node_by_id     = {n["id"]: n for n in nodes}
    _model         = _get_model(embed_model_id)

    # ── AST-only search index ──────────────────────────────────────────────────
    # Corpus BM25 chỉ trên node code; thiếu source thì dùng label+name+file
    bm25_nodes = [n for n in nodes if n["label"] not in ("Document", "DocChunk")]
    corpus_ast = []
    for n in tqdm(bm25_nodes, desc=f"[{_stem}] tokenize AST", unit="node", leave=False):
        text = n.get("source", "")
        if not text or len(text.strip()) < 5:
            text = f"{n.get('label','')} {n.get('name','')} {n.get('file','')}"
        corpus_ast.append(_tokenize_code(text))
    _t_tok_ast = time.perf_counter()
    bm25_ast = BM25Okapi(corpus_ast)
    _t_bm25_ast = time.perf_counter()

    # Ma trận vector + map id→hàng cho cả index BM25 và vector
    vec_nodes  = [n for n in bm25_nodes if n.get("embedding") is not None]
    vec_matrix = np.array([n["embedding"] for n in vec_nodes])
    vi_of      = {nd["id"]: i for i, nd in enumerate(vec_nodes)}
    bi_of      = {nd["id"]: i for i, nd in enumerate(bm25_nodes)}
    _t_vec = time.perf_counter()

    # ── Full search index (AST nodes + DocChunk) ───────────────────────────────
    # Thay vì tách từ lại toàn bộ AST (rất chậm), ta tận dụng lại corpus_ast
    doc_nodes = [n for n in nodes if n["label"] == "DocChunk"]
    corpus_doc = []
    for n in tqdm(doc_nodes, desc=f"[{_stem}] tokenize doc", unit="chunk", leave=False):
        text = f"{n.get('heading', '')} {n.get('text', '')}".strip()
        if not text or len(text.strip()) < 5:
            text = f"{n.get('label','')} {n.get('name','')}"
        corpus_doc.append(_tokenize_code(text))
        
    full_nodes = bm25_nodes + doc_nodes
    corpus_full = corpus_ast + corpus_doc
    _t_tok_doc = time.perf_counter()
    bm25_full = BM25Okapi(corpus_full)
    _t_bm25_full = time.perf_counter()

    full_vec_nodes  = [n for n in full_nodes if n.get("embedding") is not None]
    full_vec_matrix = np.array([n["embedding"] for n in full_vec_nodes])
    fvi_of          = {nd["id"]: i for i, nd in enumerate(full_vec_nodes)}
    fbi_of          = {nd["id"]: i for i, nd in enumerate(full_nodes)}
    _t_fvec = time.perf_counter()

    # ── Adjacency lists (danh sách kề có trọng số) ─────────────────────────────
    _AST_TYPES     = {"Calls", "Inherits", "Defines", "Imports"}
    _BRIDGE_LABELS = {"DocChunk"}
    # Trọng số theo loại cạnh: ngữ nghĩa code cao, cấu trúc/hub thấp, cầu doc trung bình.
    # _hop_coverage bỏ qua trọng số (giữ tương thích); _hop_ranked dùng để cho điểm.
    _EDGE_W = {"Calls": 1.0, "Inherits": 0.9, "Defines": 0.5, "Imports": 0.4,
               "Documents": 0.8, "HasChunk": 0.5, "References": 0.5}
    # fwd = xuôi (source→target), bwd = ngược — BFS chỉ đi cạnh code (AST);
    # cạnh Documents gom vào doc2code làm bảng chỉ đường chunk → code node.
    fwd_ast,  bwd_ast  = defaultdict(list), defaultdict(list)
    doc2code = defaultdict(list)
    for edge in gdata["edges"]:
        w = _EDGE_W.get(edge["type"], 0.5)
        if edge["type"] in _AST_TYPES:
            fwd_ast[edge["source"]].append((edge["target"], w))
            bwd_ast[edge["target"]].append((edge["source"], w))
        elif edge["type"] == "Documents":
            doc2code[edge["source"]].append(edge["target"])
    _t_adj = time.perf_counter()

    # Tên hiển thị của node (Function/Class dùng name, DocChunk dùng heading)
    def _node_name(nd):
        return nd.get("name") or nd.get("heading", "")

    # Lấy chỉ số top-k điểm cao nhất (argpartition cho nhanh khi k nhỏ)
    def _topn(scores, k):
        k = min(k, len(scores))
        if k == len(scores): return np.argsort(scores)[::-1]
        idx = np.argpartition(scores, -k)[-k:]
        return idx[np.argsort(scores[idx])[::-1]]

    # ── _rrf(): Reciprocal Rank Fusion — gộp 2 bảng xếp hạng BM25 + Vector ──────
    # Node có mặt ở CẢ HAI list được cộng dồn hai số hạng → ưu tiên node vừa khớp
    # lexical (BM25) vừa liên quan ngữ nghĩa (Vector). Bất biến theo thang điểm.
    RRF_C = 60
    def _rrf(bm_order, bm_id, v_order, v_id, keep_label=None):
        bm_rank = {bm_id(i): r for r, i in enumerate(bm_order)}
        v_rank  = {v_id(i):  r for r, i in enumerate(v_order)}
        cand = set(bm_rank) | set(v_rank)
        if keep_label is not None:
            cand = {nid for nid in cand if node_by_id.get(nid, {}).get("label") == keep_label}
        scored = []
        for nid in cand:
            s = 0.0
            if nid in bm_rank: s += 1.0 / (RRF_C + bm_rank[nid])
            if nid in v_rank:  s += 1.0 / (RRF_C + v_rank[nid])
            scored.append((nid, s))
        scored.sort(key=lambda x: -x[1])
        return scored

    # ── search(): hybrid trên node code (BM25 + vector) ────────────────────────
    def search(query, n=5, method="hybrid"):
        """Hybrid search on AST nodes only."""
        bm_sc = bm25_ast.get_scores(_tokenize_code(query))
        q_emb = _model.encode([_embed_query(query)], normalize_embeddings=True, convert_to_numpy=True)
        v_sim = (vec_matrix @ q_emb.T).squeeze()

        # method="bm25"/"vector": chỉ 1 tín hiệu
        if method == "bm25":
            return [
                {"id": bm25_nodes[i]["id"], "name": _node_name(bm25_nodes[i]),
                 "label": bm25_nodes[i]["label"], "score": round(float(bm_sc[i]), 4)}
                for i in _topn(bm_sc, n) if bm_sc[i] > 0
            ]
        if method == "vector":
            return [
                {"id": vec_nodes[i]["id"], "name": _node_name(vec_nodes[i]),
                 "label": vec_nodes[i]["label"], "score": round(float(v_sim[i]), 4)}
                for i in _topn(v_sim, n)
            ]

        # hybrid: gộp 2 bảng xếp hạng BM25 + Vector bằng RRF (ưu tiên node vào cả hai)
        k = min(50, len(bm25_nodes))
        bm25_top_idx = _topn(bm_sc, k)
        vec_top_idx  = _topn(v_sim, min(k, len(vec_nodes)))
        ranked = _rrf(bm25_top_idx, lambda i: bm25_nodes[i]["id"],
                      vec_top_idx,  lambda i: vec_nodes[i]["id"])
        return [
            {"id": nid, "name": _node_name(node_by_id[nid]),
             "label": node_by_id[nid]["label"], "score": round(sc, 4)}
            for nid, sc in ranked[:n] if nid in node_by_id
        ]

    # ── search_full(): hybrid trên node code + DocChunk ────────────────────────
    def search_full(query, n=5):
        """Hybrid search on AST nodes + DocChunk nodes."""
        bm_sc = bm25_full.get_scores(_tokenize_code(query))
        q_emb = _model.encode([_embed_query(query)], normalize_embeddings=True, convert_to_numpy=True)
        v_sim = (full_vec_matrix @ q_emb.T).squeeze()

        k = min(50, len(full_nodes))
        bm25_top_idx = _topn(bm_sc, k)
        vec_top_idx  = _topn(v_sim, min(k, len(full_vec_nodes)))
        ranked = _rrf(bm25_top_idx, lambda i: full_nodes[i]["id"],
                      vec_top_idx,  lambda i: full_vec_nodes[i]["id"])
        return [
            {"id": nid, "name": _node_name(node_by_id[nid]),
             "label": node_by_id[nid]["label"], "score": round(sc, 4)}
            for nid, sc in ranked[:n] if nid in node_by_id
        ]

    # ── search_doc(): xếp hạng TRỰC TIẾP trong tập DocChunk có cầu Documents ──
    # Chunk cạnh tranh với nhau (không với code) — hết nghẽn top-50 làm đói seed.
    _chunk_ids = [n["id"] for n in doc_nodes if n["id"] in doc2code]
    _chunk_bi  = {n["id"]: len(bm25_nodes) + i for i, n in enumerate(doc_nodes)}

    def search_doc(query, n=5):
        """Hybrid RRF xếp hạng nội bộ các DocChunk có cạnh Documents."""
        if not _chunk_ids:
            return []
        q_emb, _, bm_full = _q_sig(query)
        bm = [float(bm_full[_chunk_bi[cid]]) for cid in _chunk_ids]
        vs = [float(full_vec_matrix[fvi_of[cid]] @ q_emb) if cid in fvi_of else 0.0
              for cid in _chunk_ids]
        idxs = range(len(_chunk_ids))
        r_bm = {i: r for r, i in enumerate(sorted(idxs, key=lambda i: -bm[i]))}
        r_vs = {i: r for r, i in enumerate(sorted(idxs, key=lambda i: -vs[i]))}
        scored = sorted(
            ((cid, 1.0 / (RRF_C + r_bm[i]) + 1.0 / (RRF_C + r_vs[i]))
             for i, cid in enumerate(_chunk_ids)),
            key=lambda x: -x[1])
        return [{"id": cid, "name": _node_name(node_by_id[cid]),
                 "label": "DocChunk", "score": round(sc, 4)}
                for cid, sc in scored[:n]]

    # ── doc_votes(): chunk liên quan BỎ PHIẾU cho code node nó document ───────
    # Doc evidence vào RANKING (bảng doc-rank của RRF) thay vì lái traversal →
    # hub API không kéo cả vùng lân cận vào coverage như cơ chế doc-seed cũ.
    def doc_votes(query, n_doc=5, spill=0.5, max_fanout=20):
        votes = {}
        for ch in search_doc(query, n=n_doc):
            for tgt in doc2code.get(ch["id"], []):
                nd = node_by_id.get(tgt)
                if not nd or nd["label"] not in ("Function", "Class"):
                    continue
                v = ch["score"] * _EDGE_W["Documents"]
                if v > votes.get(tgt, 0.0):
                    votes[tgt] = v
                # lan 1 bước Calls (API được document → helper nó gọi), né hub
                nbrs = fwd_ast.get(tgt, [])
                if len(nbrs) <= max_fanout:
                    for nxt, _w in nbrs:
                        nd2 = node_by_id.get(nxt)
                        if nd2 and nd2["label"] in ("Function", "Class"):
                            nv = v * spill
                            if nv > votes.get(nxt, 0.0):
                                votes[nxt] = nv
        return votes

    # ── _hop_coverage(): BFS thuần từ seed, trả MỌI node Function/Class chạm tới ─
    def _hop_coverage(seeds_fn, fwd, bwd, n_seeds, depth):
        """Pure BFS from seeds — no decay, no rerank, returns ALL reachable Function/Class nodes."""
        seeds = seeds_fn(n_seeds)
        if not seeds: return []
        visited = {s["id"]: 0 for s in seeds}
        queue = deque((s["id"], 0) for s in seeds)
        # Lan tỏa theo cả cạnh xuôi và ngược tới độ sâu `depth`
        while queue:
            nid, d = queue.popleft()
            if d >= depth: continue
            for nxt, _ in fwd.get(nid, []) + bwd.get(nid, []):
                if nxt not in visited:
                    visited[nxt] = d + 1
                    queue.append((nxt, d + 1))
        # Bỏ Module/Document/DocChunk, chỉ trả Function/Class kèm số hop
        out = []
        for nid, hop in visited.items():
            nd = node_by_id.get(nid)
            if not nd or nd["label"] in ("Module", "Document", "DocChunk"): continue
            out.append({"id": nid, "name": _node_name(nd), "label": nd["label"],
                        "file": nd.get("file", ""), "hop": hop, "score": 0.0})
        return out

    # Coverage chỉ qua cạnh code (seed = search hybrid)
    def hop_coverage_ast_fn(query, n_seeds=5, depth=2):
        return _hop_coverage(lambda n: search(query, n=n, method="hybrid"),
                             fwd_ast, bwd_ast, n_seeds, depth)

    # Coverage Full = coverage AST ∪ node có doc-vote (vote KHÔNG kéo hàng xóm);
    # node chỉ-có-vote gán hop=0 (mức seed, không qua traversal).
    def hop_coverage_full_fn(query, n_seeds=5, depth=2, n_doc=5):
        cov = _hop_coverage(lambda n: search(query, n=n, method="hybrid"),
                            fwd_ast, bwd_ast, n_seeds, depth)
        seen = {r["id"] for r in cov}
        for nid in doc_votes(query, n_doc=n_doc):
            if nid not in seen:
                nd = node_by_id[nid]
                cov.append({"id": nid, "name": _node_name(nd), "label": nd["label"],
                            "file": nd.get("file", ""), "hop": 0, "score": 0.0})
        return cov

    # ── _q_sig(): memo tín hiệu query — encode + BM25 scan chỉ 1 lần / query ──
    _q_cache = {}
    def _q_sig(query):
        sig = _q_cache.get(query)
        if sig is None:
            q_emb = _model.encode([_embed_query(query)], normalize_embeddings=True,
                                  convert_to_numpy=True)[0]
            toks = _tokenize_code(query)
            sig = (q_emb, bm25_ast.get_scores(toks), bm25_full.get_scores(toks))
            _q_cache[query] = sig
        return sig

    # ── _ce_scores(): cross-encoder chấm (query, node) — memo theo query+node ─
    # Coverage giữa các config (depth/n_seeds) trùng nhau nhiều → mỗi node chỉ
    # chấm 1 lần/query dù matrix gọi hop ranking 12 lần/instance.
    _ce_q_cache = {}
    def _ce_scores(query, cand):
        cache = _ce_q_cache.setdefault(query, {})
        todo = [nid for nid in cand if nid not in cache]
        if todo:
            q = _embed_query(query)
            pairs = []
            for nid in todo:
                nd = node_by_id.get(nid, {})
                text = (nd.get("source")
                        or f"{nd.get('label','')} {nd.get('name','')} {nd.get('file','')}")[:4000]
                pairs.append((q, text))
            scores = _get_ce().predict(pairs, batch_size=64, show_progress_bar=False)
            for nid, s in zip(todo, scores):
                cache[nid] = float(s)
        return {nid: cache[nid] for nid in cand}

    # ── _hop_ranked(): graph SINH ứng viên, cross-encoder XẾP HẠNG ─────────────
    # CE chấm trực tiếp từng cặp trên MỘT thang điểm nên không còn gì để RRF
    # trộn — RRF chỉ còn trong baseline Hybrid và search_doc (sinh ứng viên doc).
    def _hop_ranked(seeds_fn, fwd, bwd, n_seeds, depth, query, votes=None):
        seeds = seeds_fn(n_seeds)
        if not seeds:
            return []
        # BFS 2 chiều thuần — chỉ để xác định tập reachable + số hop
        hopd = {s["id"]: 0 for s in seeds}
        queue = deque((s["id"], 0) for s in seeds)
        while queue:
            nid, d = queue.popleft()
            if d >= depth:
                continue
            for nxt, _w in fwd.get(nid, []) + bwd.get(nid, []):
                if nxt not in hopd:
                    hopd[nxt] = d + 1
                    queue.append((nxt, d + 1))
        # Candidates = Function/Class trong coverage ∪ node có doc-vote
        # (vote đi vào một mình, không kéo hàng xóm như seed BFS)
        votes = votes or {}
        cand = {nid for nid in hopd
                if node_by_id.get(nid, {}).get("label") in ("Function", "Class")}
        cand |= set(votes)
        cand = list(cand)
        if not cand:
            return []
        ce = _ce_scores(query, cand)
        out = [{"id": nid, "name": _node_name(node_by_id[nid]),
                "label": node_by_id[nid]["label"],
                "file": node_by_id[nid].get("file", ""),
                "hop": hopd.get(nid, 0), "score": round(ce[nid], 4)}
               for nid in cand]
        out.sort(key=lambda r: -r["score"])
        return out

    # AST-hop có xếp hạng: seed = search hybrid, đi qua cạnh code
    def hop_ast_ranked_fn(query, n_seeds=5, depth=2):
        return _hop_ranked(lambda n: search(query, n=n, method="hybrid"),
                           fwd_ast, bwd_ast, n_seeds, depth, query)

    # Full-hop = AST-hop (cùng seed, cùng BFS) + doc_votes MỞ RỘNG tập ứng viên —
    # chênh lệch recall so với AST-hop = đóng góp của lớp doc ở khâu sinh ứng viên.
    def hop_full_ranked_fn(query, n_seeds=5, depth=2, n_doc=5):
        return _hop_ranked(lambda n: search(query, n=n, method="hybrid"),
                           fwd_ast, bwd_ast, n_seeds, depth, query,
                           votes=doc_votes(query, n_doc=n_doc))

    # BFS đơn giản trả khoảng cách hop từ tập seed (dùng cho callers/callees)
    def _bfs(seeds, adj, depth):
        dist = {s: 0 for s in seeds}
        q = deque((s, 0) for s in seeds)
        while q:
            cur, d = q.popleft()
            if d >= depth: continue
            for nxt, _ in adj.get(cur, []):
                if nxt not in dist:
                    dist[nxt] = d + 1
                    q.append((nxt, d + 1))
        return dist

    # Người gọi (đi ngược) / bị gọi (đi xuôi) của 1 node trong đồ thị code
    def callers_fn(node_id, depth=2):
        return _bfs([node_id], bwd_ast, depth)

    def callees_fn(node_id, depth=2):
        return _bfs([node_id], fwd_ast, depth)

    # bridged=0 → Full-hop ≡ AST-hop, biết trước khi nhìn bảng matrix
    n_bridged = sum(1 for n in doc_nodes if n["id"] in doc2code)
    print(f"[{_stem}] indices: ast={len(bm25_nodes)} chunks={len(doc_nodes)} "
          f"(bridged={n_bridged}) vec={vec_matrix.shape} | {time.perf_counter()-_t0:.1f}s", flush=True)
    print(f"[{_stem}] index timing: load={_t_load-_t0:.1f}s "
          f"tok_ast={_t_tok_ast-_t_load:.1f}s bm25_ast={_t_bm25_ast-_t_tok_ast:.1f}s "
          f"vec={_t_vec-_t_bm25_ast:.1f}s tok_doc={_t_tok_doc-_t_vec:.1f}s "
          f"bm25_full={_t_bm25_full-_t_tok_doc:.1f}s vec_full={_t_fvec-_t_bm25_full:.1f}s "
          f"adj={_t_adj-_t_fvec:.1f}s", flush=True)

    # Gói toàn bộ hàm + dữ liệu vào dict kết quả, cache lại rồi trả về
    result = {
        "nodes": nodes, "node_by_id": node_by_id,
        "search": search, "search_full": search_full, "search_doc": search_doc,
        "doc_votes": doc_votes,
        "documented": {t for ts in doc2code.values() for t in ts},
                "hop_coverage_ast": hop_coverage_ast_fn,
        "hop_coverage_full": hop_coverage_full_fn,
        "hop_ast_ranked": hop_ast_ranked_fn,
        "hop_full_ranked": hop_full_ranked_fn,
        "callers": callers_fn, "callees": callees_fn,
    }
    _INDEX_CACHE[output_file] = result
    return result


load_graph_nodes() / load_graph_indices() ready


In [ ]:
import unidiff

# ── get_oracle_nodes(): đáp án vàng mức NODE — node nào giao với vùng patch sửa ─
def get_oracle_nodes(patch_text, graph_nodes):
    """
    Parse a unified diff patch → map changed hunks to graph nodes by line overlap.
    Returns set of node IDs whose [start_line, end_line] overlaps any changed hunk.

    Inspired by CodeRAG-Bench's get_oracle_filenames(), extended to function/class granularity
    using the start_line / end_line fields already stored on every RepoGraph node.
    """
    # Bước 1: từ patch lấy {file: [(dòng_đầu, dòng_cuối) của từng hunk bị sửa]}
    changed = {}  # file path → [(hunk_start, hunk_end), ...]
    try:
        for pf in unidiff.PatchSet(patch_text):
            fpath = pf.source_file.split("a/", 1)[-1]
            ranges = []
            for h in pf:
                if h.source_length > 0:
                    ranges.append((h.source_start, h.source_start + h.source_length - 1))
                else:  # pure insertion → anchor to enclosing line (gán vào hàm bao quanh)
                    ranges.append((h.source_start, h.source_start))
            if ranges:
                changed[fpath] = ranges
    except Exception:
        return set()

    # Bước 2: node Function/Class có [start_line,end_line] GIAO với hunk → gold
    gold_ids = set()
    for node in graph_nodes:
        if node["label"] not in ("Function", "Class"):
            continue
        node_file  = node.get("file", "")
        node_start = node.get("start_line") or 0
        node_end   = node.get("end_line")   or 0
        for diff_file, ranges in changed.items():
            # khớp file theo đuôi đường dẫn (hai chiều, tránh lệch tiền tố)
            if not (node_file.endswith(diff_file) or diff_file.endswith(node_file)):
                continue
            for (s, e) in ranges:
                if node_start <= e and node_end >= s:   # điều kiện giao khoảng
                    gold_ids.add(node["id"])
                    break
    return gold_ids


# ── get_oracle_files(): đáp án vàng mức FILE — lấy thẳng basename file bị đổi ──
def get_oracle_files(patch_text):
    """File-level gold: basename các file bị đổi LẤY THẲNG TỪ PATCH — độc lập với
    việc node có overlap hay không (bền hơn cho metric file, vốn là headline)."""
    files = set()
    try:
        for pf in unidiff.PatchSet(patch_text):
            path = pf.path                       # đã bỏ tiền tố a//b/, xử lý /dev/null
            if path and path != "/dev/null":
                files.add(path.split("/")[-1])
    except Exception:
        pass
    return files


get_oracle_nodes() ready — patch-based oracle extraction


In [ ]:
import pandas as pd
from collections import defaultdict


# ── evaluate_retrieval(): tính metric rank-aware tại budget k (node & file riêng) ─
def evaluate_retrieval(retrieved_list, gold_ids, node_by_id=None, k=None, gold_files=None):
    """Rank-aware metrics tại budget k (None = cả list). Node & file tính độc lập.
    - Đếm GOLD DISTINCT (recall ≤ 1).
    - gold_files: truyền từ patch (get_oracle_files). Nếu None → suy từ gold_ids (fallback).
    - File của kết quả tra qua node_by_id theo 'id' → dùng được cho cả search lẫn hop."""
    node_by_id = node_by_id or {}
    ret      = retrieved_list[:k] if k else retrieved_list   # cắt theo budget k
    n_ret    = len(ret)
    total_gt = len(gold_ids)
    # Không truyền gold_files → suy từ gold node (fallback)
    if gold_files is None:
        gold_files = {node_by_id[g]["file"].split("/")[-1]
                      for g in gold_ids if g in node_by_id and node_by_id[g].get("file")}

    # Duyệt kết quả theo thứ hạng: gom file trả về, đếm node trúng, ghi rank trúng đầu tiên
    covered, hit_pos, first_rank, ret_files = set(), 0, None, set()
    for rank, r in enumerate(ret, start=1):
        rid = r.get("id")
        f = node_by_id.get(rid, {}).get("file", "").split("/")[-1]
        if f:
            ret_files.add(f)
        if rid in gold_ids:
            covered.add(rid); hit_pos += 1
            if first_rank is None:
                first_rank = rank

    # Tổng hợp các chỉ số: recall/precision/MRR/hit/all cho cả node lẫn file
    hit_files = ret_files & gold_files
    return {
        "recall_nodes":    round(len(covered) / total_gt, 4) if total_gt else 0.0,
        "precision_nodes": round(hit_pos / n_ret, 4) if n_ret else 0.0,
        "mrr":             round(1.0 / first_rank, 4) if first_rank else 0.0,
        "hit_nodes":       int(len(covered) > 0),
        "all_nodes":       int(total_gt > 0 and len(covered) == total_gt),
        "hits_nodes":      f"{len(covered)}/{total_gt}",
        "recall_files":    round(len(hit_files) / len(gold_files), 4) if gold_files else 0.0,
        "hit_files":       int(len(hit_files) > 0),
        "all_files":       int(bool(gold_files) and gold_files.issubset(ret_files)),
        "hits_files":      f"{len(hit_files)}/{len(gold_files)}",
    }


evaluate_retrieval() defined


## 2. Build Test Data (SWE-bench)
Load SWE-bench Lite, filter Flask instances, clone the repo at the target commit, and enumerate source files.

In [ ]:
from datasets import load_dataset

# ── Nạp SWE-bench Lite, lọc các instance thuộc REPO đang xét ──────────────────
_swe_lite = load_dataset("princeton-nlp/SWE-bench_lite", split="test")
SWE_INSTANCES = [
    {
        "id":     row["instance_id"],
        "repo":   row["repo"],
        "commit": row["base_commit"],                    # commit để checkout
        "title":  row["instance_id"].split("__", 1)[-1],
        "text":   row["problem_statement"],              # dùng làm query truy hồi
        "patch":  row["patch"],                          # dùng để trích đáp án vàng
    }
    for row in _swe_lite
    if row["repo"] in REPOS
]

# Chọn 1 instance theo INSTANCE_IDX (None → dùng HEAD, không checkout)
TARGET_COMMIT = SWE_INSTANCES[INSTANCE_IDX]["commit"] if INSTANCE_IDX is not None else None
TARGET_ID     = SWE_INSTANCES[INSTANCE_IDX]["id"]     if INSTANCE_IDX is not None else "HEAD"

from collections import Counter as _Counter
_by_repo = _Counter(i["repo"] for i in SWE_INSTANCES)
print(f"Total instances : {len(SWE_INSTANCES)}  ({dict(_by_repo)})")
print(f"Instance        : {TARGET_ID}")
print(f"Commit          : {TARGET_COMMIT[:8] if TARGET_COMMIT else 'HEAD'}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/120k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/23 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Total instances : 17
Instance        : pytest-dev__pytest-11143
Commit          : 6995257c


In [ ]:
import os
from pathlib import Path
from git import Repo

# ── Clone tất cả repo trong REPOS nếu chưa có trong /content ──────────────────
for _r in REPOS:
    _p = repo_path_of(_r)
    if not Path(_p).exists():
        print(f"Cloning https://github.com/{_r}.git ...")
        Repo.clone_from(f"https://github.com/{_r}.git", _p)
print(f"{len(REPOS)} repo sẵn sàng: " + ", ".join(repo_path_of(r) for r in REPOS))

# Checkout commit của instance đã chọn (trên đúng repo của nó)
if INSTANCE_IDX is not None:
    _inst = SWE_INSTANCES[INSTANCE_IDX]
    repo = Repo(repo_path_of(_inst["repo"]))
    repo.git.checkout(_inst["commit"])
    BASE_COMMIT = repo.head.commit.hexsha[:8]
    OUTPUT_FILE = f"{_inst['repo'].split('/')[1]}_graph_{BASE_COMMIT}.json"
    print(f"Checked out : {BASE_COMMIT}  ({_inst['id']})")
    print(f"Output      : {OUTPUT_FILE}")


Cloning https://github.com/pytest-dev/pytest.git ...
Done.
Checked out : 6995257c  (pytest-dev__pytest-11143)
Remote      : https://github.com/pytest-dev/pytest.git
Commit      : 6995257c
Output      : pytest_graph_6995257c.json


## 3. Build Graphs & Statistics
Build the graph for every SWE-bench Flask instance (results are cached in memory), then print aggregate node/edge statistics.

In [ ]:
# GPU / CUDA check - run before Build Graphs to confirm embeddings use the GPU
import torch

# ── In trạng thái CUDA và thông tin GPU (nếu có) ──────────────────────────────
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name    :", torch.cuda.get_device_name(0))
    print("CUDA version   :", torch.version.cuda)
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("No GPU detected - embeddings run on CPU (slow).")
    print("Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU")

# ── Xác nhận model embedding thực sự đang nằm trên thiết bị nào ───────────────
# Which device will the embedding model actually use?
try:
    _m = _get_model()
    print("Embed model on :", next(_m.parameters()).device)
except NameError:
    print("(_get_model not defined yet - run the Globals cell first)")
except Exception as e:
    print("Model not loaded yet:", e)

In [ ]:
import json
from collections import Counter
from tqdm.auto import tqdm

# ── Build graph cho TẤT CẢ instance + gom thống kê (warm cache cho Section 4) ─
print("Building graphs for all SWE instances (first run builds + embeds; subsequent runs are cache hits)...\n")
all_stats = []
for inst in tqdm(SWE_INSTANCES, desc="Building graphs"):
    output_file = build_graph(inst["commit"], repo_path=repo_path_of(inst["repo"]))
    with open(output_file, "r", encoding="utf-8") as f:
        gdata = json.load(f)
    nodes = gdata["nodes"]
    edges = gdata["edges"]
    # Đếm node theo label, cạnh theo type, và LOC trung bình mỗi hàm
    nc = Counter(n["label"] for n in nodes)
    ec = Counter(e["type"] for e in edges)
    fl = [n["end_line"] - n["start_line"] + 1
          for n in nodes if n["label"] == "Function"
          and n.get("start_line") and n.get("end_line")]
    all_stats.append({
        "id":          inst["id"],
        "nodes":       len(nodes),
        "edges":       len(edges),
        "functions":   nc["Function"],
        "classes":     nc["Class"],
        "calls":       ec["Calls"],
        "imports":     ec["Imports"],
        "avg_func_loc": round(sum(fl) / len(fl), 1) if fl else 0,
    })
    tag = inst["id"].split("__", 1)[-1][:38]
    print(f"  [{tag:<38}]  nodes={len(nodes):3}  edges={len(edges):4}", flush=True)

# ── Aggregate statistics ──────────────────────────────────────────────────────
# In trung bình các chỉ số trên toàn bộ instance
n = len(all_stats)
sep = "=" * 56
print(f"\n{sep}")
print(f"  Instances  : {n}")
for key, label in [
    ("nodes",        "Avg nodes      "),
    ("edges",        "Avg edges      "),
    ("functions",    "Avg functions  "),
    ("classes",      "Avg classes    "),
    ("calls",        "Avg Call edges "),
    ("imports",      "Avg Import edges"),
    ("avg_func_loc", "Avg func LOC   "),
]:
    print(f"  {label}: {sum(s[key] for s in all_stats) / n:.1f}")
print(f"\nAll {n} graphs built and cached — Section 4 runs entirely from memory cache.")


Building graphs for all SWE instances (first run builds + embeds; subsequent runs are cache hits)...

[6995257c] checked out
[6995257c] docs: 1755 nodes | HasChunk=1529  Documents=203  References=0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. Experiments
Run retrieval experiments on the same SWE-bench Flask instances using two independent systems, then compare in Section 5.

### 4.1 Our Pipeline
Mỗi SWE-bench instance: build/load graph (Section 3), chạy **5 method đều ranked** — BM25, Vector, Hybrid, AST-hop, Full-hop — rồi chấm bằng metric nhận biết thứ hạng ở **cùng budget k** (headline k=10). Oracle: node-level (patch line-overlap) + file-level (file đổi trong patch).

In [ ]:
# ── Các trục điều khiển matrix (baseline ↔ hop TÁCH RIÊNG) ───────────────────
# TOP_K   : budget top-k KẾT QUẢ — CHỈ cho baseline search (BM25/Vector/Hybrid)
# N_SEEDS : số code-seed khởi đầu hop — cho AST-hop/Full-hop
# M_DOC   : số doc-seed THÊM vào (CHỈ Full-hop); m=0 ≡ AST-hop (không quét lại)
# K_HOP   : độ sâu BFS (k-hop)   — cho AST-hop/Full-hop
TOP_K_GRID   = [5, 10, 20]   # baseline: quét budget
N_SEEDS_GRID = [5, 10]       # hop: quét số code-seed
M_DOC_GRID   = [5, 10]       # Full-hop: quét số doc-seed (m=0 = AST-hop)
K_HOP_GRID   = [1, 2]        # hop: quét độ sâu

# Giá trị baseline (điểm tham chiếu khi hiển thị / khi quét trục khác)
TOP_K   = 10
N_SEEDS = 5
M_DOC   = 5
K_HOP   = 2

# Hằng số dùng chung cho matrix + mọi cell kết quả/visualize (định nghĩa 1 nơi)
BASELINES    = ["BM25", "Vector", "Hybrid"]
METHOD_ORDER = BASELINES + ["AST-hop", "Full-hop"]
METHOD_COLOR = {"BM25": "#6090c8", "Vector": "#9060c8", "Hybrid": "#e0a020",
                "AST-hop": "#fd8d3c", "Full-hop": "#40c040"}

print(f"TOP_K_GRID={TOP_K_GRID} | N_SEEDS_GRID={N_SEEDS_GRID} | M_DOC_GRID={M_DOC_GRID} | K_HOP_GRID={K_HOP_GRID}")
print(f"baseline ref: TOP_K={TOP_K}, N_SEEDS={N_SEEDS}, M_DOC={M_DOC}, K_HOP={K_HOP}")

In [ ]:
# ── Xóa cache index để đảm bảo dùng ĐÚNG code search/index hiện tại ───────────
# load_graph_indices() cache kết quả KÈM closure (search/search_full/hop...) theo
# _INDEX_CACHE. Sau khi sửa cell 16 (RRF, đổi seeding...), phải clear thì index mới
# được build lại — nếu không, Section 4/5 chạy bằng closure CŨ và thay đổi không có hiệu lực.
# (Không đụng _EMB_CACHE/_MODEL_CACHE → embedding không phải tính lại.)
_INDEX_CACHE.clear()
print(f"_INDEX_CACHE cleared ({len(_INDEX_CACHE)} entries) — indices sẽ build lại từ code hiện tại.")

In [ ]:
# Verify doc layer before running 4.2 — check that Documents edges exist in loaded graph
# ── Sanity check: graph có cạnh Documents (doc→code) không? ───────────────────
_sample = SWE_INSTANCES[0]
_f = build_graph(_sample["commit"], repo_path=repo_path_of(_sample["repo"]))
with open(_f) as _fp:
    _g = json.load(_fp)

# Đếm node theo label và cạnh theo type
from collections import Counter
edge_counts = Counter(e["type"] for e in _g["edges"])
node_counts = Counter(n["label"] for n in _g["nodes"])

print("Node labels:", dict(node_counts))
print("Edge types: ", dict(edge_counts))
print()

# Nếu Documents = 0 → lớp doc không đóng góp gì cho Full-hop; in nguyên nhân khả dĩ
n_doc = edge_counts.get("Documents", 0)
if n_doc == 0:
    print("⚠️  No Documents edges found — doc layer will NOT contribute to hop_full.")
    print("   Possible causes:")
    print("   1. Old cached graph (schema != simple+doc) — delete /content/flask_graph*.json and rerun")
    print("   2. RST name resolution failed — check build_graph output for 'Documents=0'")
else:
    print(f"✓  {n_doc} Documents edges found — hop_full should differ from hop_ast")
    # Show a sample (in vài cạnh doc→code minh họa)
    doc_edges = [e for e in _g["edges"] if e["type"] == "Documents"][:5]
    for e in doc_edges:
        src_label = "chunk" if e["source"].startswith("chunk:") else "?"
        tgt = _g_nodes_by_id = {n["id"]: n for n in _g["nodes"]}
        tgt_node = tgt.get(e["target"], {})
        print(f"   {e['source'][:50]}  →  {tgt_node.get('name','?')} ({tgt_node.get('file','?')})")

In [ ]:
# ── Doc ceiling: doc-vote (chunk → code node) có trúng gold không? ────────────
# Nếu "trúng gold tại vote" thấp mà Full-hop vẫn ≈ AST-hop → cầu doc→code còn
# thưa/sai (xem build_name_index / resolve_doc_links), không phải lỗi ranking.
for inst in SWE_INSTANCES:
    idx   = load_graph_indices(build_graph(inst["commit"], repo_path=repo_path_of(inst["repo"])))
    gt    = get_oracle_nodes(inst["patch"], idx["nodes"])
    votes = idx["doc_votes"](inst["text"], n_doc=10)
    hits  = gt & set(votes)
    print(f"{inst['id'].split('__')[-1]:32s} doc-vote={len(votes):3d}  "
          f"trúng gold ngay tại vote={len(hits)}/{len(gt)}")


In [ ]:
# ── Thí nghiệm matrix (thống nhất: mọi method → ranked list → top-k) ──────────
# baseline (BM25/Vector/Hybrid): search → top-k qua TOP_K_GRID.
# hop (AST-hop/Full-hop): coverage → RERANK (cross-encoder) → top-k qua TOP_K_GRID.
#   Giữ CovRecall (recall trên FULL coverage = TRẦN) + CovSize (số node coverage thô)
#   để thấy rerank+cắt mất bao nhiêu. Mọi method giờ có recall@k/precision@k/F1@k/MRR/tokens@k.
# Size/Tokens tính SAU dedup node lồng nhau (Class chứa method → tính Class 1 lần).
import pandas as pd
import numpy as np

_BM_METHOD = {"BM25": "bm25", "Vector": "vector", "Hybrid": "hybrid"}

# ── Bộ đếm token (tiktoken nếu có, không thì ước lượng len//4) ────────────────
try:
    import tiktoken
    _ENC = tiktoken.get_encoding("cl100k_base")
    def _tok(s): return len(_ENC.encode(s, disallowed_special=()))
    _TOK_MODE = "tiktoken/cl100k_base"
except Exception:
    def _tok(s): return max(1, len(s) // 4) if s else 0
    _TOK_MODE = "ước lượng len//4"
print("Token counter:", _TOK_MODE)

_ntok = {}
def _node_tokens(nid, nbi):
    if nid not in _ntok:
        nd = nbi.get(nid, {})
        _ntok[nid] = _tok(nd.get("source") or nd.get("text") or "")
    return _ntok[nid]

def _dedupe_contained(ids, nbi):
    """Bỏ node nằm TRỌN trong node khác (cùng file, [start,end] bao nhau) → giữ node bao ngoài."""
    withln, noln = [], []
    for nid in ids:
        nd = nbi.get(nid, {})
        f, s, e = nd.get("file"), nd.get("start_line"), nd.get("end_line")
        if f and s is not None and e is not None:
            withln.append((nid, f, s, e))
        else:
            noln.append(nid)
    withln.sort(key=lambda x: -(x[3] - x[2]))
    kept, ranges = [], []
    for nid, f, s, e in withln:
        if any(rf == f and rs <= s and re >= e for (rf, rs, re) in ranges):
            continue
        kept.append(nid); ranges.append((f, s, e))
    return kept + noln

def _size_tokens(items, nbi):
    ids = [it["id"] for it in items]
    kept = _dedupe_contained(ids, nbi)
    return len(kept), len(ids), int(sum(_node_tokens(nid, nbi) for nid in kept))

def _f1(r, p):
    return round(2 * r * p / (r + p), 4) if (r + p) else 0.0

def _counts(ev):
    hn, gn = (int(x) for x in ev["hits_nodes"].split("/"))
    hf, gf = (int(x) for x in ev["hits_files"].split("/"))
    return hn, gn, hf, gf

def _row(iid, group, method, tk, ns, md, kh, ev, size, size_raw, tokens, docseed, mrr,
         cov_recall=np.nan, cov_size=np.nan):
    hn, gn, hf, gff = _counts(ev)
    return {
        "Instance": iid, "Group": group, "Method": method,
        "TOP_K": tk, "N_SEEDS": ns, "M_DOC": md, "K_HOP": kh,
        "Recall": ev["recall_nodes"], "Precision": ev["precision_nodes"],
        "F1": _f1(ev["recall_nodes"], ev["precision_nodes"]), "F-Recall": ev["recall_files"],
        "HitN": hn, "GoldN": gn, "HitF": hf, "GoldF": gff,
        "Hit": ev["hit_nodes"], "All": ev["all_nodes"],
        "Size": size, "SizeRaw": size_raw, "Tokens": tokens,
        "CovRecall": cov_recall, "CovSize": cov_size, "DocSeed": docseed, "MRR": mrr,
    }

def _eval_topk(ranked_ids, gt, nbi, gf, tk):
    """Chấm top-tk của danh sách đã rerank: ev + (size dedup, size raw, tokens)."""
    top = [{"id": c} for c in ranked_ids[:tk]]
    ev = evaluate_retrieval(top, gt, node_by_id=nbi, k=tk, gold_files=gf)
    sz, szr, tok = _size_tokens(top, nbi)
    return ev, sz, szr, tok

# RUNS: artefact per-instance cho các cell phân tích phía sau (detail, hop dist)
# dùng lại — tránh mỗi cell tự build_graph/get_oracle một lần nữa.
RUNS = {}

def add_hit_cols(df):
    """Thêm cột node_hit/file_hit dạng 'đúng/tổng' cho bảng hiển thị."""
    df = df.copy()
    df["node_hit"] = df["HitN"].astype(int).astype(str) + "/" + df["GoldN"].astype(int).astype(str)
    df["file_hit"] = df["HitF"].astype(int).astype(str) + "/" + df["GoldF"].astype(int).astype(str)
    return df

matrix_rows = []
for inst in SWE_INSTANCES:
    output_file = build_graph(inst["commit"], repo_path=repo_path_of(inst["repo"]))
    idx = load_graph_indices(output_file)
    nbi = idx["node_by_id"]
    gt  = get_oracle_nodes(inst["patch"], idx["nodes"])
    gf  = get_oracle_files(inst["patch"])
    q   = inst["text"]
    iid = inst["id"].split("__", 1)[-1]
    RUNS[iid] = {"inst": inst, "output_file": output_file, "gt": gt, "gf": gf, "q": q}
    kmax = max(TOP_K_GRID)

    # Cờ degenerate: GOLD=0 → loại instance khỏi trung bình khi đọc kết quả;
    # DOC_VOTE=0 → Full-hop trùng AST-hop, không nói lên gì về lớp doc
    _n_dv = len(idx["doc_votes"](q, n_doc=max(M_DOC_GRID)))
    _flags = []
    if not gt:
        _flags.append("GOLD=0")
    if _n_dv == 0:
        _flags.append("DOC_VOTE=0 (Full-hop ≡ AST-hop)")
    print(f"[{iid}] gold={len(gt)} files={len(gf)} doc_vote={_n_dv}"
          + ("  ⚠ " + " | ".join(_flags) if _flags else ""), flush=True)

    # ── Baselines: search → chấm tại từng TOP_K ──
    for m in BASELINES:
        ret = idx["search"](q, n=kmax, method=_BM_METHOD[m])
        for tk in TOP_K_GRID:
            sub = ret[:tk]
            ev  = evaluate_retrieval(ret, gt, node_by_id=nbi, k=tk, gold_files=gf)
            sz, szr, tok = _size_tokens(sub, nbi)
            matrix_rows.append(_row(iid, "search", m, tk, np.nan, np.nan, np.nan,
                                    ev, sz, szr, tok, np.nan, ev["mrr"]))

    # ── AST-hop: Sử dụng kết quả lan truyền đồ thị THỰC SỰ ──
    for ns in N_SEEDS_GRID:
        for kh in K_HOP_GRID:
            # 1. Đo lường trần coverage (CovRecall) y như cũ
            cov = idx["hop_coverage_ast"](q, n_seeds=ns, depth=kh)
            ev_cov = evaluate_retrieval(cov, gt, node_by_id=nbi, k=None, gold_files=gf)
            cr, cs = ev_cov["recall_nodes"], len(cov)
            
            # 2. Xếp hạng bằng cross-encoder trên coverage
            ranked = idx["hop_ast_ranked"](q, n_seeds=ns, depth=kh)
            ranked_ids = [n["id"] for n in ranked]
            
            for tk in TOP_K_GRID:
                ev, sz, szr, tok = _eval_topk(ranked_ids, gt, nbi, gf, tk)
                matrix_rows.append(_row(iid, "hop", "AST-hop", tk, ns, 0, kh,
                                        ev, sz, szr, tok, 0, ev["mrr"], cov_recall=cr, cov_size=cs))

    # ── Full-hop: Đảm bảo có trộn doc-seed và lấy điểm đồ thị THỰC SỰ ──
    for ns in N_SEEDS_GRID:
        for md in M_DOC_GRID:
            n_doc_seed = len(idx["doc_votes"](q, n_doc=md))
            for kh in K_HOP_GRID:
                # 1. Đo lường trần coverage
                cov = idx["hop_coverage_full"](q, n_seeds=ns, depth=kh, n_doc=md)
                ev_cov = evaluate_retrieval(cov, gt, node_by_id=nbi, k=None, gold_files=gf)
                cr, cs = ev_cov["recall_nodes"], len(cov)
                
                # 2. Xếp hạng bằng cross-encoder (ứng viên = coverage ∪ doc-vote)
                ranked = idx["hop_full_ranked"](q, n_seeds=ns, depth=kh, n_doc=md)
                ranked_ids = [n["id"] for n in ranked]
                
                for tk in TOP_K_GRID:
                    ev, sz, szr, tok = _eval_topk(ranked_ids, gt, nbi, gf, tk)
                    matrix_rows.append(_row(iid, "hop", "Full-hop", tk, ns, md, kh,
                                            ev, sz, szr, tok, n_doc_seed, ev["mrr"],
                                            cov_recall=cr, cov_size=cs))

matrix_df = pd.DataFrame(matrix_rows)
_hopdf = matrix_df[matrix_df["Group"] == "hop"]
print(f"matrix_df: {len(matrix_df)} rows  ({len(SWE_INSTANCES)} inst)")
print(f"Hop — rerank+cắt mất bao nhiêu recall: mean CovRecall(trần)={_hopdf['CovRecall'].mean():.3f}  "
      f"vs recall@k(sống sót)={_hopdf['Recall'].mean():.3f}  | coverage size mean={_hopdf['CovSize'].mean():.0f}")
matrix_df.head()

### 4.2 Kết quả matrix — tổng hợp + chi tiết từng instance

Bảng mean theo nhóm (baseline theo `TOP_K`, hop theo `(N_SEEDS,K_HOP)`) kèm **số đếm hit** (`đúng/tổng`) và **số doc-seed**; sau đó là breakdown **từng instance**.

In [ ]:
# ── Bảng tóm tắt matrix (mọi method đo cùng recall@k / tokens@k) ──────────────
# Hop: Recall = recall@k (SỐNG SÓT sau rerank+cắt); CovRecall = trần (recall trên full coverage).
# node_hit/file_hit = TỔNG hit "đúng/tổng". Breakdown đầy đủ theo (N,M,K_HOP,TOP_K) ở cell detail.
import pandas as pd

# Loại instance oracle rỗng (GOLD=0) — mọi method đều 0, chỉ kéo mean vô nghĩa
_valid = matrix_df[matrix_df["GoldN"] > 0]
_excluded = sorted(set(matrix_df["Instance"]) - set(_valid["Instance"]))
if _excluded:
    print(f"(loại {len(_excluded)} instance GOLD=0 khỏi tổng hợp: {', '.join(_excluded)})\n")
n_inst = _valid["Instance"].nunique()
_base = _valid[_valid["Group"] == "search"]
_hop  = _valid[_valid["Group"] == "hop"]

# ── Baseline: recall@k / precision@k / F1@k + tokens@k theo TOP_K ──
b = _base.groupby(["Method", "TOP_K"]).agg(
    Recall=("Recall", "mean"), Precision=("Precision", "mean"), F1=("F1", "mean"),
    FRecall=("F-Recall", "mean"), MRR=("MRR", "mean"), Tokens=("Tokens", "mean"),
    HitN=("HitN", "sum"), GoldN=("GoldN", "sum"), HitF=("HitF", "sum"), GoldF=("GoldF", "sum"),
).round(2)
b = add_hit_cols(b)
print(f"=== Baseline — recall@k/precision@k/F1@k + tokens@k theo TOP_K (trên {n_inst} instance) ===")
print(b[["Recall", "Precision", "F1", "FRecall", "MRR", "node_hit", "file_hit", "Tokens"]])

# ── Hop headline: recall@k (sống sót) vs CovRecall (trần) theo (Method, TOP_K) ──
h = _hop.groupby(["Method", "TOP_K"]).agg(
    Recall=("Recall", "mean"), CovRecall=("CovRecall", "mean"), Precision=("Precision", "mean"),
    F1=("F1", "mean"), FRecall=("F-Recall", "mean"),
    MRR=("MRR", "mean"), Size=("Size", "mean"), Tokens=("Tokens", "mean"),
    CovSize=("CovSize", "mean"),
    HitN=("HitN", "sum"), GoldN=("GoldN", "sum"), HitF=("HitF", "sum"), GoldF=("GoldF", "sum"),
).round(2)
h = add_hit_cols(h)
print("\n=== Hop — recall@k(sống sót) vs CovRecall(trần) + size/tokens theo (Method, TOP_K) ===")
print(h[["Recall", "CovRecall", "Precision", "F1", "FRecall", "MRR", "Size", "Tokens", "CovSize", "node_hit"]])
print("(mean gộp qua N_SEEDS/M_DOC/K_HOP — xem cell detail cho breakdown từng cấu hình)")
print("(node_hit/file_hit là tổng qua SỐ CONFIG KHÁC NHAU giữa method — chỉ so tỉ lệ, không so số tuyệt đối)")

# ── Chẩn đoán: DocSeed = số code-node nhận doc-vote theo m chunk yêu cầu ──────
print("\n=== Số code-node nhận doc-vote theo M_DOC yêu cầu ===")
print(_hop.groupby(["Method", "M_DOC"])["DocSeed"].mean().round(2).to_frame("avg_doc_vote"))

# ── Δ POC: hai con số headline của thí nghiệm ─────────────────────────────────
# (1) Graph vs Flat: AST-hop − Hybrid (flat mạnh nhất), khớp theo (Instance, TOP_K)
# (2) Doc vs AST   : Full-hop − AST-hop, khớp theo (Instance, N_SEEDS, K_HOP, TOP_K)
_ast_cfg = _hop[_hop.Method == "AST-hop"].groupby(["Instance", "N_SEEDS", "K_HOP", "TOP_K"])["Recall"].mean()
_full_cfg = _hop[_hop.Method == "Full-hop"].groupby(["Instance", "N_SEEDS", "K_HOP", "TOP_K"])["Recall"].mean()
_d_doc = (_full_cfg - _ast_cfg).dropna()

_ast_tk = _hop[_hop.Method == "AST-hop"].groupby(["Instance", "TOP_K"])["Recall"].mean()
_hyb_tk = _base[_base.Method == "Hybrid"].set_index(["Instance", "TOP_K"])["Recall"]
_d_graph = (_ast_tk - _hyb_tk).dropna()

print("\n=== Δ POC — recall@k (mean Δ | win/tie/loss theo instance-config) ===")
print(f"  Graph vs Flat (AST-hop − Hybrid)  : {_d_graph.mean():+.3f}"
      f"  |  {(_d_graph > 0).sum()}/{(_d_graph == 0).sum()}/{(_d_graph < 0).sum()}")
print(f"  Doc   vs AST  (Full-hop − AST-hop): {_d_doc.mean():+.3f}"
      f"  |  {(_d_doc > 0).sum()}/{(_d_doc == 0).sum()}/{(_d_doc < 0).sum()}")

In [ ]:
# ── Chi tiết TỪNG INSTANCE (đọc từ RUNS + matrix_df — không build lại gì) ─────
# (1) Header: title, gold files, DOC-COVERAGE = số gold node được tài liệu hóa (trần doc).
# (2) Bảng metric theo cấu hình (baseline: k{TOP_K}; hop: n·m·h·k) — recall@k (sống sót)
#     kèm CovRecall (trần coverage) + Size/SizeRaw/Tokens/Hit/All/MRR.
# (3) Bản đồ GOLD × METHOD @baseline: mỗi gold node method nào GIAO được (sau rerank→top-k).
import pandas as pd
from IPython.display import display

detail = add_hit_cols(matrix_df)
detail["cfg"] = [f"k{int(t)}" if g == "search" else f"n{int(n)}·m{int(m)}·h{int(k)}·k{int(t)}"
                 for g, t, n, m, k in zip(detail.Group, detail.TOP_K.fillna(0), detail.N_SEEDS.fillna(0),
                                          detail.M_DOC.fillna(0), detail.K_HOP.fillna(0))]
detail["Method"] = pd.Categorical(detail["Method"], categories=METHOD_ORDER, ordered=True)
detail = detail.sort_values(["Instance", "Method", "N_SEEDS", "M_DOC", "K_HOP", "TOP_K"])

_cols = ["Recall", "CovRecall", "Precision", "F1", "F-Recall", "node_hit", "file_hit",
         "Size", "SizeRaw", "Tokens", "Hit", "All", "MRR", "DocSeed"]

print(f"Chi tiết {detail['Instance'].nunique()} instance  (bản đồ @baseline: TOP_K={TOP_K}, "
      f"N_SEEDS={N_SEEDS}, M_DOC={M_DOC}, K_HOP={K_HOP}):")
for iid, part in detail.groupby("Instance", sort=False):
    run = RUNS[iid]
    idx = load_graph_indices(run["output_file"])          # memory cache hit — rẻ
    nbi, gt, gfiles, q = idx["node_by_id"], run["gt"], run["gf"], run["q"]
    documented = idx["documented"]
    gold_docd  = gt & documented

    # (1) Header
    _title = run["inst"].get("title", "")
    print(f"\n{'='*70}\n  {iid}" + (f"   —   {_title}" if _title else ""))
    print(f"  gold files ({len(gfiles)}): {', '.join(sorted(gfiles))}")
    print(f"  gold nodes: {len(gt)}   |   ĐƯỢC TÀI LIỆU HÓA (trần doc): {len(gold_docd)}/{len(gt)}"
          + (f"  → {[nbi.get(g,{}).get('name','?') for g in sorted(gold_docd)]}" if gold_docd else ""))

    # (2) Bảng metric theo cấu hình
    display(part.set_index(["Method", "cfg"])[_cols])

    # (3) Bản đồ GOLD × METHOD @baseline (sau rerank→top-k = đúng cái LLM sẽ thấy)
    ret_ids = {
        "BM25":     {r["id"] for r in idx["search"](q, n=TOP_K, method="bm25")},
        "Vector":   {r["id"] for r in idx["search"](q, n=TOP_K, method="vector")},
        "Hybrid":   {r["id"] for r in idx["search"](q, n=TOP_K, method="hybrid")},
        "AST-hop":  {n["id"] for n in idx["hop_ast_ranked"](q, n_seeds=N_SEEDS, depth=K_HOP)[:TOP_K]},
        "Full-hop": {n["id"] for n in idx["hop_full_ranked"](q, n_seeds=N_SEEDS, depth=K_HOP, n_doc=M_DOC)[:TOP_K]},
    }
    rows = []
    for g in sorted(gt):
        nd = nbi.get(g, {})
        row = {"gold_node": nd.get("name", "?"),
               "file": (nd.get("file", "") or "").split("/")[-1],
               "doc'd": "✓" if g in documented else ""}
        for mth in METHOD_ORDER:
            row[mth] = "✓" if g in ret_ids[mth] else "·"
        rows.append(row)
    if rows:
        display(pd.DataFrame(rows).set_index("gold_node"))


## 5. Compare & Visualize

Đọc từ `matrix_df` (không chạy lại retrieval):

1. **Fig 1 — recall@k & tokens@k theo TOP_K** — cả 5 method trên cùng budget; nét đứt = trần coverage của hop. Trả lời *graph vs flat*.
2. **Fig 2 — Full-hop theo M_DOC** (0 = AST-hop) — recall/F1/size/tokens khi thêm doc-router seed. Trả lời *doc vs AST-only*.


In [ ]:
# ── Visualize matrix (thống nhất recall@k / tokens@k cho cả 5 method) ─────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ORDER, CMAP = METHOD_ORDER, METHOD_COLOR
hop = matrix_df[matrix_df["Group"] == "hop"]

# ── Fig 1: recall@k & tokens@k theo TOP_K (cả 5 method); nét đứt = trần coverage của hop ──
rec    = matrix_df.pivot_table(index="TOP_K", columns="Method", values="Recall",  aggfunc="mean")
tok    = matrix_df.pivot_table(index="TOP_K", columns="Method", values="Tokens",  aggfunc="mean")
covrec = hop.pivot_table(index="TOP_K", columns="Method", values="CovRecall", aggfunc="mean")
print("=== recall@k theo TOP_K (mean; hop gộp qua N/M/K_HOP) ===")
display(rec.reindex(columns=[m for m in ORDER if m in rec.columns]).round(3))

fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.6))
for m in ORDER:
    if m in rec.columns:
        ax1.plot(rec.index, rec[m], marker="o", label=m, color=CMAP[m], lw=2)
for m in ("AST-hop", "Full-hop"):
    if m in covrec.columns:
        ax1.plot(covrec.index, covrec[m], ls="--", lw=1.3, alpha=.7, color=CMAP[m],
                 label=f"{m} trần(Cov)")
ax1.set_xlabel("TOP_K"); ax1.set_ylabel("Recall@k"); ax1.set_ylim(0, 1); ax1.set_xticks(TOP_K_GRID)
ax1.set_title("recall@k theo TOP_K  (nét đứt = trần coverage của đồ thị)")
ax1.grid(alpha=.25, ls="--"); ax1.set_axisbelow(True); ax1.legend(fontsize=7)

for m in ORDER:
    if m in tok.columns:
        ax2.plot(tok.index, tok[m], marker="o", label=m, color=CMAP[m], lw=2)
ax2.set_xlabel("TOP_K"); ax2.set_ylabel("Tokens@k"); ax2.set_xticks(TOP_K_GRID)
ax2.set_title("tokens@k theo TOP_K  (chi phí context)")
ax2.grid(alpha=.25, ls="--"); ax2.set_axisbelow(True); ax2.legend(fontsize=8)
fig1.suptitle("Thống nhất: mọi method → ranked → top-k  |  recall@k vs trần, và chi phí token",
              fontweight="bold")
plt.tight_layout(); plt.savefig("fig_recall_topk.png", dpi=150, bbox_inches="tight"); plt.show()

# ── Fig 2: hop — metric theo M_DOC (x=0 = AST-hop) tại K_HOP & TOP_K baseline ──
hop_b = hop[(hop.K_HOP == K_HOP) & (hop.TOP_K == TOP_K)]
ms_axis = sorted(hop_b["M_DOC"].dropna().unique())      # [0, 5, 10] (0 = AST-hop)
NCOLOR  = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]
_metrics = ["Recall", "F1", "Size", "Tokens"]
_counts_metric = {"Size", "Tokens"}
print(f"\n=== Hop theo M_DOC tại (K_HOP={K_HOP}, TOP_K={TOP_K}) ===")
display(hop_b.groupby(["N_SEEDS", "M_DOC"])[["Recall", "CovRecall", "F1", "Size", "Tokens"]].mean().round(3))
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric in zip(axes2.ravel(), _metrics):
    for ci, ns in enumerate(N_SEEDS_GRID):
        vals = []
        for md in ms_axis:
            s = hop_b[(hop_b.N_SEEDS == ns) & (hop_b.M_DOC == md)][metric]
            vals.append(s.mean() if len(s) else np.nan)
        ax.plot(ms_axis, vals, marker="o", color=NCOLOR[ci % len(NCOLOR)], lw=2, label=f"N_SEEDS={ns}")
        for x, v in zip(ms_axis, vals):
            if not np.isnan(v):
                ax.annotate(f"{v:.0f}" if metric in _counts_metric else f"{v:.2f}",
                            (x, v), textcoords="offset points", xytext=(0, 5), fontsize=7, ha="center")
    ax.set_xticks(ms_axis); ax.set_xlabel("M_DOC — số chunk yêu cầu (0 = AST-hop)")
    ax.set_title(f"Hop {metric} theo doc-seed"); ax.grid(alpha=.25, ls="--"); ax.set_axisbelow(True)
    if metric not in _counts_metric:
        ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
fig2.suptitle(f"Full-hop = AST-hop + doc-vote (m chunk mở rộng ứng viên cho CE) @K_HOP={K_HOP}, TOP_K={TOP_K} — recall@k có tăng? token phình?",
              fontweight="bold")
plt.tight_layout(); plt.savefig("fig_hop_mdoc.png", dpi=150, bbox_inches="tight"); plt.show()

### 5.1 Phân tích bổ sung — Hop distribution (#1) & Precision–Recall (#4)

Đọc từ index (coverage) và `matrix_df`:
- **#1** — gold node trúng nằm ở hop nào (hop 0 = seed/search, hop≥1 = nhờ traversal), tại baseline `(N_SEEDS, K_HOP)`.
- **#4** — đánh đổi Precision / Recall / F1: baseline @`TOP_K` vs hop @`(N_SEEDS, K_HOP)`.

In [ ]:
# ── #1: Phân bố HOP của gold trúng — coverage tại baseline (N_SEEDS, K_HOP) ───
# Gold node lấy được nằm ở hop 0 (seed search; Full-hop thêm node có doc-vote) hay hop≥1?
# Chênh lệch hop0 giữa Full-hop và AST-hop = gold trúng TRỰC TIẾP nhờ lớp doc.
# Đọc từ RUNS (matrix đã build) + index cache — không build lại gì.
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

HOP_FN = {"AST-hop": "hop_coverage_ast", "Full-hop": "hop_coverage_full"}
HOPS   = ["AST-hop", "Full-hop"]
hop_hits   = {m: defaultdict(int) for m in HOPS}
gold_total = 0
for iid, run in RUNS.items():
    idx = load_graph_indices(run["output_file"])   # memory cache hit — rẻ
    gt, q = run["gt"], run["q"]
    gold_total += len(gt)
    for m in HOPS:
        ret = idx[HOP_FN[m]](q, n_seeds=N_SEEDS, depth=K_HOP)   # full coverage @baseline
        for r in ret:
            if r["id"] in gt:
                hop_hits[m][r.get("hop", 0)] += 1

# ── Bảng: gold hit theo hop + %recall + %nhờ traversal (hop≥1) ───────────────
hop_levels = list(range(K_HOP + 1))
print(f"=== #1 — Gold hit theo hop (coverage, N_SEEDS={N_SEEDS}, K_HOP={K_HOP}, tổng gold={gold_total}) ===")
print(f"{'Method':<10} " + " ".join(f"hop{h:>2}" for h in hop_levels) + "   total  %recall  %via_traversal")
for m in HOPS:
    counts = [hop_hits[m].get(h, 0) for h in hop_levels]
    tot, via = sum(counts), sum(counts[1:])
    rec = tot / gold_total if gold_total else 0.0
    vtr = via / tot if tot else 0.0
    print(f"{m:<10} " + " ".join(f"{c:>5}" for c in counts) + f"   {tot:>5}   {rec:>6.2f}   {vtr:>6.2f}")

# ── Hình: stacked bar theo tầng hop ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.2))
hop_colors = [plt.cm.YlOrRd(0.25 + 0.6 * h / max(1, K_HOP)) for h in hop_levels]
x = np.arange(len(HOPS)); bottom = np.zeros(len(HOPS))
for hi, h in enumerate(hop_levels):
    vals = np.array([hop_hits[m].get(h, 0) for m in HOPS], dtype=float)
    bars = ax.bar(x, vals, bottom=bottom, width=0.55, color=hop_colors[hi],
                  edgecolor="white", label=f"hop {h}" + (" (seed/doc-vote)" if h == 0 else ""))
    for b, v in zip(bars, vals):
        if v > 0:
            ax.text(b.get_x() + b.get_width() / 2, b.get_y() + v / 2, int(v),
                    ha="center", va="center", fontsize=8, fontweight="bold")
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(HOPS)
ax.set_ylabel("Số gold node trúng (tập coverage)")
ax.set_title(f"#1 — Gold hit theo hop  (N_SEEDS={N_SEEDS}, K_HOP={K_HOP})")
ax.legend(title="Tầng lan truyền"); ax.yaxis.grid(True, alpha=.25, ls="--"); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig("fig_hop_dist.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ── #4: Đánh đổi Precision–Recall / F1 — baseline @TOP_K vs hop @baseline ─────
# Điểm tham chiếu: baseline @TOP_K; AST-hop @(N_SEEDS,K_HOP,m=0); Full-hop @(N_SEEDS,M_DOC,K_HOP).
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base_at = matrix_df[(matrix_df.Group == "search") & (matrix_df.TOP_K == TOP_K)]
hop_at  = matrix_df[(matrix_df.Group == "hop") & (matrix_df.N_SEEDS == N_SEEDS)
                    & (matrix_df.K_HOP == K_HOP) & (matrix_df.M_DOC.isin([0, M_DOC]))
                    & (matrix_df.TOP_K == TOP_K)]
ORDER, CMAP = METHOD_ORDER, METHOD_COLOR

pr = pd.concat([base_at, hop_at]).groupby("Method")[["Recall", "Precision", "F1"]].mean().reindex(ORDER)
print(f"=== #4 — P/R/F1 (baseline TOP_K={TOP_K} | AST-hop N={N_SEEDS},h={K_HOP} | "
      f"Full-hop N={N_SEEDS},m={M_DOC},h={K_HOP}) ===")
print(pr.round(3))

fig, (axb, axs) = plt.subplots(1, 2, figsize=(13, 4.4))
metrics, mc = ["Recall", "Precision", "F1"], ["#4c78a8", "#f58518", "#54a24b"]
xp = np.arange(len(ORDER)); w = 0.26
for j, (mt, c) in enumerate(zip(metrics, mc)):
    vals = pr[mt].values.astype(float)
    bars = axb.bar(xp + (j - 1) * w, vals, width=w, color=c, edgecolor="white", label=mt)
    for b, v in zip(bars, vals):
        if not np.isnan(v):
            axb.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=7)
axb.set_xticks(xp); axb.set_xticklabels(ORDER, rotation=30, ha="right", fontsize=9)
axb.set_ylim(0, 1); axb.set_title("Recall / Precision / F1"); axb.legend(fontsize=8)
axb.yaxis.grid(True, alpha=.25, ls="--"); axb.set_axisbelow(True)

for m in ORDER:
    if m in pr.index and not np.isnan(pr.loc[m, "Recall"]):
        axs.scatter(pr.loc[m, "Recall"], pr.loc[m, "Precision"], s=90,
                    color=CMAP[m], edgecolor="black", zorder=3)
        axs.annotate(m, (pr.loc[m, "Recall"], pr.loc[m, "Precision"]),
                     textcoords="offset points", xytext=(6, 4), fontsize=8)
rr = np.linspace(0.01, 1, 200)
for f1v in (0.2, 0.4, 0.6, 0.8):
    pp = f1v * rr / (2 * rr - f1v)
    pp[(pp < 0) | (pp > 1)] = np.nan
    axs.plot(rr, pp, ls=":", color="gray", lw=.8)
    axs.annotate(f"F1={f1v}", (1.0, f1v / (2 - f1v)), fontsize=7, color="gray")
axs.set_xlim(0, 1); axs.set_ylim(0, 1)
axs.set_xlabel("Recall (node)"); axs.set_ylabel("Precision (node)")
axs.set_title("Precision–Recall (nền = iso-F1)"); axs.grid(alpha=.2)
plt.tight_layout(); plt.savefig("fig_pr_f1.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# xóa tất các file graph JSON
# (đang tắt để tránh vô tình xóa; bỏ comment dòng dưới khi muốn dọn /content)
# !rm -f /content/*.json